In [1]:
# ============================================================
# CELL 1: IMPORT LIBRARIES
# ============================================================

import os
import time
import random
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

In [1]:
# ============================================================
# CELL 2: CORRECTED CONFIGURATION
# ============================================================

SEED = 42

DATASET_PATH = r"C:\Users\SIRPI\Downloads\theatrical_movement_dataset.csv"

TARGET_COL = "expressiveness_class"
ID_COL = "id"

OUTPUT_DIR = "APSO_G_STDL_Corrected_Output"

# ------------------------------------------------------------
# IMPORTANT REVIEWER CORRECTION
# ------------------------------------------------------------
# The IoT-AME records are treated as independent observations.
# No artificial temporal sequence is constructed.
# No anatomical joint graph is assumed.
# ------------------------------------------------------------

USE_TEMPORAL_SEQUENCE = False

PCA_VARIANCE = 0.95

BATCH_SIZE = 32
FINAL_EPOCHS = 80

# APSO configuration
APSO_PARTICLES = 30
APSO_ITERATIONS = 100

# Hyperparameter bounds
LR_MIN = 1e-4
LR_MAX = 1e-3

HIDDEN_MIN = 32
HIDDEN_MAX = 128

DROPOUT_MIN = 0.20
DROPOUT_MAX = 0.50

WEIGHT_DECAY_MIN = 1e-6
WEIGHT_DECAY_MAX = 1e-3

EARLY_STOPPING_PATIENCE = 12

os.makedirs(OUTPUT_DIR, exist_ok=True)


def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 75)
print("CORRECTED APSO-G-STDL CONFIGURATION")
print("=" * 75)

print("Random Seed                 :", SEED)
print("Temporal sequence modeling :", USE_TEMPORAL_SEQUENCE)
print("Artificial temporal window :", "DISABLED")
print("Anatomical joint graph     :", "DISABLED")
print("PCA variance retained      :", PCA_VARIANCE)
print("Batch size                 :", BATCH_SIZE)
print("Final epochs               :", FINAL_EPOCHS)
print("APSO particles             :", APSO_PARTICLES)
print("APSO iterations            :", APSO_ITERATIONS)
print("Device                     :", device)

CORRECTED APSO-G-STDL CONFIGURATION
Random Seed                 : 42
Temporal sequence modeling : False
Artificial temporal window : DISABLED
Anatomical joint graph     : DISABLED
PCA variance retained      : 0.95
Batch size                 : 32
Final epochs               : 80
APSO particles             : 30
APSO iterations            : 100
Device                     : cuda


In [7]:
# ================================================================
# 2. DATA PIPELINE
# ================================================================
DATASET_PATH = r"C:\Users\SIRPI\Downloads\theatrical_movement_dataset.csv"
TARGET_COL = "expressiveness_class"
ID_COL = "id"

def load_and_prepare_dataset(path, target_col, id_col="id"):
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    from sklearn.impute import SimpleImputer
    from sklearn.decomposition import PCA

    df = pd.read_csv(path)

    if target_col not in df.columns:
        raise ValueError(
            f"Target column '{target_col}' not found. "
            f"Available columns: {list(df.columns)}"
        )

    drop_cols = [target_col]
    if id_col in df.columns:
        drop_cols.append(id_col)

    Xdf = df.drop(columns=drop_cols).select_dtypes(include=[np.number])
    y_raw = df[target_col].astype(str)

    enc = LabelEncoder()
    y = enc.fit_transform(y_raw)

    Xtr, Xtmp, ytr, ytmp = train_test_split(
        Xdf.values, y, test_size=0.30,
        random_state=SEED, stratify=y
    )
    Xv, Xte, yv, yte = train_test_split(
        Xtmp, ytmp, test_size=0.50,
        random_state=SEED, stratify=ytmp
    )

    imp = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    Xtr = scaler.fit_transform(imp.fit_transform(Xtr))
    Xv = scaler.transform(imp.transform(Xv))
    Xte = scaler.transform(imp.transform(Xte))

    pca = PCA(n_components=PCA_VARIANCE, random_state=SEED)
    Xtr_pca = pca.fit_transform(Xtr)
    Xv_pca = pca.transform(Xv)
    Xte_pca = pca.transform(Xte)

    corr = np.nan_to_num(np.corrcoef(Xtr_pca, rowvar=False))
    A = np.abs(corr)
    A[A < GRAPH_THRESHOLD] = 0
    np.fill_diagonal(A, 1)
    A = A / np.maximum(A.sum(axis=1, keepdims=True), 1e-8)

    return {
        "df": df,
        "X_train": Xtr_pca,
        "X_val": Xv_pca,
        "X_test": Xte_pca,
        "y_train": ytr,
        "y_val": yv,
        "y_test": yte,
        "classes": enc.classes_,
        "pca": pca,
        "adjacency": A
    }

Pipeline definition validated.
No artificial temporal window is created.
PCA and graph construction are training-set fitted.


In [6]:
# ================================================================
# 1. REPRODUCIBLE EXPERIMENT CONFIGURATION
# ================================================================
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Reviewer-corrected APSO settings
APSO_PARTICLES = 30
APSO_ITERATIONS = 100
C1 = 1.5
C2 = 1.5
W_MAX = 0.9
W_MIN = 0.4

HIDDEN_MIN = 32
HIDDEN_MAX = 128
DROPOUT_MIN = 0.20
DROPOUT_MAX = 0.50
LR_MIN = 1e-4
LR_MAX = 1e-3
WD_MIN = 1e-6
WD_MAX = 1e-3

PCA_VARIANCE = 0.95
GRAPH_THRESHOLD = 0.30
TEMPORAL_WINDOW = None

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

set_seed(SEED)

config_table = pd.DataFrame([{
    "Parameter": "Random seed",
    "Value": SEED
},{
    "Parameter": "Train / Validation / Test",
    "Value": "70% / 15% / 15%"
},{
    "Parameter": "APSO particles",
    "Value": APSO_PARTICLES
},{
    "Parameter": "APSO iterations",
    "Value": APSO_ITERATIONS
},{
    "Parameter": "c1",
    "Value": C1
},{
    "Parameter": "c2",
    "Value": C2
},{
    "Parameter": "Inertia",
    "Value": f"{W_MAX} -> {W_MIN}"
},{
    "Parameter": "Hidden dimension",
    "Value": f"{HIDDEN_MIN}-{HIDDEN_MAX}"
},{
    "Parameter": "Dropout",
    "Value": f"{DROPOUT_MIN}-{DROPOUT_MAX}"
},{
    "Parameter": "Learning rate",
    "Value": f"{LR_MIN}-{LR_MAX}"
},{
    "Parameter": "Weight decay",
    "Value": f"{WD_MIN}-{WD_MAX}"
},{
    "Parameter": "PCA criterion",
    "Value": PCA_VARIANCE
},{
    "Parameter": "Graph",
    "Value": "Absolute correlation of PCA components"
},{
    "Parameter": "Temporal window",
    "Value": "Not used"
}])

display(config_table)

                         Parameter                         Value
0                    Random seed                            42
1              Train / Validation / Test                70% / 15% / 15%
2                    APSO particles                            30
3                   APSO iterations                           100
4                                c1                            1.5
5                                c2                            1.5
6                           Inertia                    0.9 -> 0.4
7                  Hidden dimension                         32-128
8                           Dropout                    0.2-0.5
9                    Learning rate                 0.0001-0.001
10                  Weight decay                 1e-06-0.001
11                   PCA criterion                           0.95
12                           Graph   Absolute correlation of PCA components
13                  Temporal window                       Not used


In [3]:
# Separate Features and Target

drop_cols = [TARGET_COL]

if ID_COL in df.columns:
    drop_cols.append(ID_COL)

X_df = df.drop(columns=drop_cols)
X_df = X_df.select_dtypes(include=[np.number])

if X_df.shape[1] == 0:
    raise ValueError("No numeric feature columns found.")

y_raw = df[TARGET_COL].astype(str)

feature_columns = list(X_df.columns)

print("\nSelected Feature Columns:")
for i, col in enumerate(feature_columns, start=1):
    print(f"{i}. {col}")

print("\nNumber of Features:", len(feature_columns))


Selected Feature Columns:
1. joint_angle
2. velocity
3. acceleration
4. angular_velocity
5. trajectory_length
6. movement_smoothness
7. energy_level
8. symmetry_index
9. rhythm_score
10. spatial_variation

Number of Features: 10


In [4]:
# Encode Target Labels

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_raw)

class_names = list(label_encoder.classes_)
num_classes = len(class_names)

print("\nEncoded Classes:")
for i, cls in enumerate(class_names):
    print(f"{i} -> {cls}")


Encoded Classes:
0 -> High
1 -> Low
2 -> Medium


In [5]:
X_train_raw, X_temp_raw, y_train_raw, y_temp_raw = train_test_split(
    X_df.values,
    y_encoded,
    test_size=0.30,
    random_state=SEED,
    stratify=y_encoded
)

X_val_raw, X_test_raw, y_val_raw, y_test_raw = train_test_split(
    X_temp_raw,
    y_temp_raw,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp_raw
)

print("DATA SPLIT COMPLETED")

DATA SPLIT COMPLETED


In [2]:
# ============================================================
# CELL 6: TRAIN / VALIDATION / TEST SPLIT
# ============================================================

X_train_raw, X_temp_raw, y_train_raw, y_temp_raw = train_test_split(
    X_df.values,
    y_encoded,
    test_size=0.30,
    random_state=SEED,
    stratify=y_encoded
)

X_val_raw, X_test_raw, y_val_raw, y_test_raw = train_test_split(
    X_temp_raw,
    y_temp_raw,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp_raw
)

print("=" * 75)
print("DATA SPLIT")
print("=" * 75)

print("Training   :", X_train_raw.shape)
print("Validation :", X_val_raw.shape)
print("Testing    :", X_test_raw.shape)

print("\nTotal observations:",
      len(X_train_raw) +
      len(X_val_raw) +
      len(X_test_raw))

DATA SPLIT
Training   : (1764, 10)
Validation : (378, 10)
Testing    : (378, 10)

Total observations: 2520


In [6]:
imputer = SimpleImputer(strategy="median")

X_train_imp = imputer.fit_transform(X_train_raw)
X_val_imp = imputer.transform(X_val_raw)
X_test_imp = imputer.transform(X_test_raw)

print("\nStep Completed: Missing Value Imputation")


def wavelet_denoise_1d(signal, wavelet="db4", level=2):
    """
    Wavelet denoising for a 1D feature signal.
    """
    signal = np.asarray(signal, dtype=np.float64)

    if len(signal) < 8:
        return signal

    wavelet_obj = pywt.Wavelet(wavelet)
    max_level = pywt.dwt_max_level(data_len=len(signal), filter_len=wavelet_obj.dec_len)
    used_level = min(level, max_level)

    if used_level < 1:
        return signal

    coeffs = pywt.wavedec(signal, wavelet, level=used_level, mode="symmetric")

    detail_coeff = coeffs[-1]
    sigma = np.median(np.abs(detail_coeff)) / 0.6745 if len(detail_coeff) > 0 else 0

    if sigma == 0:
        return signal

    threshold = sigma * np.sqrt(2 * np.log(len(signal)))

    denoised_coeffs = [coeffs[0]]
    for c in coeffs[1:]:
        denoised_coeffs.append(pywt.threshold(c, threshold, mode="soft"))

    reconstructed = pywt.waverec(denoised_coeffs, wavelet, mode="symmetric")
    return reconstructed[:len(signal)]


def wavelet_denoise_matrix(X_matrix):
    """
    Applies wavelet denoising column-wise.
    """
    X_out = np.zeros_like(X_matrix, dtype=np.float64)

    for j in range(X_matrix.shape[1]):
        X_out[:, j] = wavelet_denoise_1d(X_matrix[:, j])

    return X_out


X_train_wt = wavelet_denoise_matrix(X_train_imp)
X_val_wt = wavelet_denoise_matrix(X_val_imp)
X_test_wt = wavelet_denoise_matrix(X_test_imp)

print("Step Completed: Wavelet Transform Denoising")


scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_wt)
X_val_scaled = scaler.transform(X_val_wt)
X_test_scaled = scaler.transform(X_test_wt)

print("Step Completed: Z-score Normalization")

def create_temporal_segments(X_data, y_data, window_size=1, step_size=1):

    X_segments = []
    y_segments = []

    n = len(X_data)

    if window_size < 1:
        raise ValueError("window_size must be >= 1")

    if n < window_size:
        raise ValueError("Dataset split is smaller than window_size.")

    for start in range(0, n - window_size + 1, step_size):
        end = start + window_size
        center_index = start + window_size // 2

        X_segments.append(X_data[start:end])
        y_segments.append(y_data[center_index])

    return np.array(X_segments, dtype=np.float32), np.array(y_segments, dtype=np.int64)


X_train_seg, y_train = create_temporal_segments(
    X_train_scaled,
    y_train_raw,
    window_size=WINDOW_SIZE,
    step_size=STEP_SIZE
)

X_val_seg, y_val = create_temporal_segments(
    X_val_scaled,
    y_val_raw,
    window_size=WINDOW_SIZE,
    step_size=STEP_SIZE
)

X_test_seg, y_test = create_temporal_segments(
    X_test_scaled,
    y_test_raw,
    window_size=WINDOW_SIZE,
    step_size=STEP_SIZE
)

print("Step Completed: Temporal Segmentation")


Step Completed: Missing Value Imputation
Step Completed: Wavelet Transform Denoising
Step Completed: Z-score Normalization
Step Completed: Temporal Segmentation


In [3]:
# ============================================================
# CELL 8: OBSERVATION-LEVEL REPRESENTATION
# ============================================================

# IMPORTANT:
# The available IoT-AME records are treated as independent
# movement observations.
#
# Consecutive dataframe rows are NOT interpreted as consecutive
# temporal frames.
#
# Therefore:
#   - no temporal window is created
#   - no artificial sequence is created
#   - no frame-to-frame transition is calculated

X_train_obs = X_train_scaled.astype(np.float32)
X_val_obs = X_val_scaled.astype(np.float32)
X_test_obs = X_test_scaled.astype(np.float32)

y_train = y_train_raw.astype(np.int64)
y_val = y_val_raw.astype(np.int64)
y_test = y_test_raw.astype(np.int64)

print("=" * 75)
print("OBSERVATION-LEVEL REPRESENTATION")
print("=" * 75)

print("Temporal sequencing          : DISABLED")
print("Artificial temporal windows : DISABLED")
print("Consecutive rows = frames   : NO")
print("Observation-level learning  : ENABLED")

print("\nTraining observations :", X_train_obs.shape)
print("Validation observations:", X_val_obs.shape)
print("Testing observations  :", X_test_obs.shape)

OBSERVATION-LEVEL REPRESENTATION
Temporal sequencing          : DISABLED
Artificial temporal windows : DISABLED
Consecutive rows = frames   : NO
Observation-level learning  : ENABLED

Training observations : (1764, 10)
Validation observations: (378, 10)
Testing observations  : (378, 10)


In [8]:
# PCA Feature Extraction
n_train, n_steps, n_features = X_train_seg.shape

X_train_2d = X_train_seg.reshape(-1, n_features)
X_val_2d = X_val_seg.reshape(-1, n_features)
X_test_2d = X_test_seg.reshape(-1, n_features)

pca = PCA(n_components=PCA_VARIANCE, random_state=SEED)

X_train_pca_2d = pca.fit_transform(X_train_2d)
X_val_pca_2d = pca.transform(X_val_2d)
X_test_pca_2d = pca.transform(X_test_2d)

pca_features = X_train_pca_2d.shape[1]

X_train_pca = X_train_pca_2d.reshape(X_train_seg.shape[0], n_steps, pca_features)
X_val_pca = X_val_pca_2d.reshape(X_val_seg.shape[0], n_steps, pca_features)
X_test_pca = X_test_pca_2d.reshape(X_test_seg.shape[0], n_steps, pca_features)

print("Step Completed: PCA Feature Extraction")
print("PCA Components Retained:", pca_features)
print("Explained Variance:", round(np.sum(pca.explained_variance_ratio_), 4))

Step Completed: PCA Feature Extraction
PCA Components Retained: 10
Explained Variance: 0.95


In [4]:
# ============================================================
# CELL 10: FEATURE-SPACE GRAPH CONSTRUCTION
# ============================================================

def build_adjacency_matrix(X_data):

    # X_data shape:
    # [observations, PCA feature nodes]

    num_nodes = X_data.shape[1]

    if num_nodes == 1:

        adjacency = np.ones(
            (1, 1),
            dtype=np.float32
        )

    else:

        correlation = np.corrcoef(
            X_data.T
        )

        correlation = np.nan_to_num(
            correlation,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

        adjacency = np.abs(correlation)

        # self-loops
        np.fill_diagonal(
            adjacency,
            1.0
        )

    # Symmetric normalization
    degree = adjacency.sum(axis=1)

    degree_inv_sqrt = np.diag(
        1.0 / np.sqrt(degree + 1e-8)
    )

    adjacency_normalized = (
        degree_inv_sqrt
        @ adjacency
        @ degree_inv_sqrt
    )

    return torch.tensor(
        adjacency_normalized,
        dtype=torch.float32
    )


adjacency_matrix = build_adjacency_matrix(
    X_train_pca
).to(device)

print("=" * 75)
print("FEATURE-SPACE GRAPH")
print("=" * 75)

print("Graph nodes               :", pca_features)
print("Node interpretation       : PCA-derived movement features")
print("Anatomical joints         : NOT USED")
print("Temporal edges            : NOT USED")
print("Edge definition           : Absolute feature correlation")
print("Adjacency matrix shape    :",
      tuple(adjacency_matrix.shape))

FEATURE-SPACE GRAPH
Graph nodes               : 8
Node interpretation       : PCA-derived movement features
Anatomical joints         : NOT USED
Temporal edges            : NOT USED
Edge definition           : Absolute feature correlation
Adjacency matrix shape    : (8, 8)


In [5]:
# ============================================================
# CELL 11: DATASET AND DATALOADER
# ============================================================

class MovementDataset(Dataset):

    def __init__(self, X_data, y_data):

        self.X_data = torch.tensor(
            X_data,
            dtype=torch.float32
        )

        self.y_data = torch.tensor(
            y_data,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.X_data)

    def __getitem__(self, index):

        return (
            self.X_data[index],
            self.y_data[index]
        )


def make_loader(
    X_data,
    y_data,
    batch_size=32,
    shuffle=False
):

    dataset = MovementDataset(
        X_data,
        y_data
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle
    )


train_loader = make_loader(
    X_train_pca,
    y_train,
    BATCH_SIZE,
    True
)

val_loader = make_loader(
    X_val_pca,
    y_val,
    BATCH_SIZE,
    False
)

test_loader = make_loader(
    X_test_pca,
    y_test,
    BATCH_SIZE,
    False
)

sample_X, sample_y = next(
    iter(train_loader)
)

print("=" * 75)
print("DATALOADER VERIFICATION")
print("=" * 75)

print("Batch X shape:", tuple(sample_X.shape))
print("Batch y shape:", tuple(sample_y.shape))

DATALOADER VERIFICATION
Batch X shape: (32, 8)
Batch y shape: (32,)


# Proposed

In [26]:
# Nodes = PCA components
# Edges = Absolute correlation between PCA components
def build_adjacency_matrix(X_data):
    """
    Builds normalized adjacency matrix from PCA feature correlations.
    """
    flat = X_data.reshape(-1, X_data.shape[-1])
    num_nodes = flat.shape[1]

    if num_nodes == 1:
        adjacency = np.ones((1, 1), dtype=np.float32)
    else:
        corr = np.corrcoef(flat.T)
        corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)

        adjacency = np.abs(corr)
        np.fill_diagonal(adjacency, 1.0)

    degree = np.sum(adjacency, axis=1)
    degree_inv_sqrt = np.diag(1.0 / np.sqrt(degree + 1e-8))
    adjacency_norm = degree_inv_sqrt @ adjacency @ degree_inv_sqrt

    return torch.tensor(adjacency_norm, dtype=torch.float32)


adjacency_matrix = build_adjacency_matrix(X_train_pca).to(device)

class MovementDataset(Dataset):
    def __init__(self, X_data, y_data):
        self.X_data = torch.tensor(X_data, dtype=torch.float32)
        self.y_data = torch.tensor(y_data, dtype=torch.long)

    def __len__(self):
        return len(self.X_data)

    def __getitem__(self, index):
        return self.X_data[index], self.y_data[index]


def make_loader(X_data, y_data, batch_size=32, shuffle=False):
    dataset = MovementDataset(X_data, y_data)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


train_loader = make_loader(X_train_pca, y_train, batch_size=BATCH_SIZE, shuffle=True)
val_loader = make_loader(X_val_pca, y_val, batch_size=BATCH_SIZE, shuffle=False)
test_loader = make_loader(X_test_pca, y_test, batch_size=BATCH_SIZE, shuffle=False)

class GraphConvolution(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(GraphConvolution, self).__init__()
        self.linear = nn.Linear(in_channels, out_channels)

    def forward(self, x, adj):
        """
        x shape   : [batch, time, nodes, channels]
        adj shape : [nodes, nodes]
        """
        x = torch.einsum("nm,btmc->btnc", adj, x)
        x = self.linear(x)
        x = F.relu(x)
        return x

class GSTDLModel(nn.Module):
    def __init__(self, num_nodes, num_classes, hidden_dim=64, dropout=0.3):
        super(GSTDLModel, self).__init__()

        self.num_nodes = num_nodes
        self.hidden_dim = hidden_dim

        self.gcn1 = GraphConvolution(1, hidden_dim)
        self.gcn2 = GraphConvolution(hidden_dim, hidden_dim)

        self.temporal_conv1 = nn.Conv1d(
            in_channels=num_nodes * hidden_dim,
            out_channels=hidden_dim,
            kernel_size=3,
            padding=1
        )

        self.temporal_conv2 = nn.Conv1d(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            kernel_size=3,
            padding=1
        )

        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, adj):
        """
        x shape: [batch, time, features]
        """
        batch_size, time_steps, num_nodes = x.shape

        # Convert PCA features into graph nodes
        x = x.unsqueeze(-1)  # [batch, time, nodes, 1]

        # Spatial learning using GNN
        x = self.gcn1(x, adj)
        x = self.gcn2(x, adj)

        # Flatten graph node embeddings for temporal learning
        x = x.reshape(batch_size, time_steps, num_nodes * self.hidden_dim)

        # Temporal CNN expects [batch, channels, time]
        x = x.permute(0, 2, 1)

        x = F.relu(self.temporal_conv1(x))
        x = F.relu(self.temporal_conv2(x))

        # Back to [batch, time, hidden]
        x = x.permute(0, 2, 1)

        # Attention mechanism
        attn_scores = self.attention(x)
        attn_weights = torch.softmax(attn_scores, dim=1)

        x = torch.sum(x * attn_weights, dim=1)

        x = self.dropout(x)
        output = self.classifier(x)

        return output

def evaluate_model(model, data_loader, adj, num_classes):
    model.eval()

    all_true = []
    all_pred = []
    all_prob = []

    with torch.no_grad():
        for batch_X, batch_y in data_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            logits = model(batch_X, adj)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_true.extend(batch_y.cpu().numpy())
            all_pred.extend(preds.cpu().numpy())
            all_prob.extend(probs.cpu().numpy())

    all_true = np.array(all_true)
    all_pred = np.array(all_pred)
    all_prob = np.array(all_prob)

    accuracy = accuracy_score(all_true, all_pred)
    precision = precision_score(all_true, all_pred, average="weighted", zero_division=0)
    recall = recall_score(all_true, all_pred, average="weighted", zero_division=0)
    f1 = f1_score(all_true, all_pred, average="weighted", zero_division=0)

    try:
        if num_classes == 2:
            auc = roc_auc_score(all_true, all_prob[:, 1])
        else:
            auc = roc_auc_score(
                all_true,
                all_prob,
                multi_class="ovr",
                average="weighted"
            )
    except Exception:
        auc = 0.0

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
        "y_true": all_true,
        "y_pred": all_pred,
        "y_prob": all_prob
    }

def train_gstdl_model(
    hidden_dim,
    dropout,
    learning_rate,
    weight_decay,
    epochs,
    train_loader,
    val_loader,
    adj,
    class_weight_tensor,
    verbose=False
):
    model = GSTDLModel(
        num_nodes=pca_features,
        num_classes=num_classes,
        hidden_dim=hidden_dim,
        dropout=dropout
    ).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weight_tensor)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=5
    )

    best_score = -1.0
    best_state = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()

            logits = model(batch_X, adj)
            loss = criterion(logits, batch_y)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        val_metrics = evaluate_model(model, val_loader, adj, num_classes)
        val_score = val_metrics["f1"]

        scheduler.step(val_score)

        if val_score > best_score:
            best_score = val_score
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            patience_counter = 0
        else:
            patience_counter += 1

        if verbose:
            print(
                f"Epoch [{epoch + 1}/{epochs}] "
                f"Loss: {total_loss / len(train_loader):.4f} "
                f"Val Acc: {val_metrics['accuracy'] * 100:.2f}% "
                f"Val F1: {val_metrics['f1'] * 100:.2f}%"
            )

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_score

classes_array = np.arange(num_classes)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes_array,
    y=y_train
)

class_weight_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

print("\nClass Weights:")
for cls_name, weight in zip(class_names, class_weights):
    print(f"{cls_name}: {weight:.4f}")


# ==========================================================
# 19. Adaptive Particle Swarm Optimization
# ==========================================================
class AdaptivePSO:
    def __init__(self, n_particles=6, max_iter=5):
        self.n_particles = n_particles
        self.max_iter = max_iter

        self.bounds = np.array([
            [32, 128],       # hidden_dim
            [0.10, 0.50],    # dropout
            [-4.0, -2.3],    # learning_rate = 1e-4 to about 5e-3
            [-6.0, -3.0]     # weight_decay = 1e-6 to 1e-3
        ], dtype=np.float64)

        self.dim = self.bounds.shape[0]

        self.positions = np.zeros((n_particles, self.dim))
        self.velocities = np.zeros((n_particles, self.dim))

        for d in range(self.dim):
            low, high = self.bounds[d]
            self.positions[:, d] = np.random.uniform(low, high, n_particles)
            self.velocities[:, d] = np.random.uniform(-0.1, 0.1, n_particles)

        self.pbest_positions = self.positions.copy()
        self.pbest_scores = np.full(n_particles, -np.inf)

        self.gbest_position = None
        self.gbest_score = -np.inf

    def decode_position(self, position):
        hidden_dim = int(round(position[0] / 8) * 8)
        hidden_dim = int(np.clip(hidden_dim, 32, 128))

        dropout = float(np.clip(position[1], 0.10, 0.50))

        learning_rate = 10 ** float(np.clip(position[2], -4.0, -2.3))
        weight_decay = 10 ** float(np.clip(position[3], -6.0, -3.0))

        return hidden_dim, dropout, learning_rate, weight_decay

    def optimize(self):
        print("APSO OPTIMIZATION STARTED")
       
        for iteration in range(self.max_iter):
            print(f"\nAPSO Iteration {iteration + 1}/{self.max_iter}")

            # Adaptive coefficients
            inertia = 0.9 - ((0.9 - 0.4) * iteration / max(1, self.max_iter - 1))
            c1 = 2.0 - (0.5 * iteration / max(1, self.max_iter - 1))
            c2 = 1.5 + (0.5 * iteration / max(1, self.max_iter - 1))

            for i in range(self.n_particles):
                hidden_dim, dropout, lr, wd = self.decode_position(self.positions[i])

                print(
                    f"\nParticle {i + 1}/{self.n_particles} | "
                    f"hidden_dim={hidden_dim}, dropout={dropout:.3f}, "
                    f"lr={lr:.6f}, wd={wd:.8f}"
                )

                set_seed(SEED + iteration + i)

                model, score = train_gstdl_model(
                    hidden_dim=hidden_dim,
                    dropout=dropout,
                    learning_rate=lr,
                    weight_decay=wd,
                    epochs=APSO_EPOCHS,
                    train_loader=train_loader,
                    val_loader=val_loader,
                    adj=adjacency_matrix,
                    class_weight_tensor=class_weight_tensor,
                    verbose=False
                )

                print(f"Validation F1 Fitness: {score * 100:.2f}%")

                if score > self.pbest_scores[i]:
                    self.pbest_scores[i] = score
                    self.pbest_positions[i] = self.positions[i].copy()

                if score > self.gbest_score:
                    self.gbest_score = score
                    self.gbest_position = self.positions[i].copy()

            # Update velocity and position
            for i in range(self.n_particles):
                r1 = np.random.rand(self.dim)
                r2 = np.random.rand(self.dim)

                cognitive = c1 * r1 * (self.pbest_positions[i] - self.positions[i])
                social = c2 * r2 * (self.gbest_position - self.positions[i])

                self.velocities[i] = inertia * self.velocities[i] + cognitive + social
                self.positions[i] = self.positions[i] + self.velocities[i]

                # Boundary control
                for d in range(self.dim):
                    low, high = self.bounds[d]
                    self.positions[i, d] = np.clip(self.positions[i, d], low, high)

            print(f"\nBest Fitness So Far: {self.gbest_score * 100:.2f}%")

        best_hidden, best_dropout, best_lr, best_wd = self.decode_position(self.gbest_position)

        return {
            "hidden_dim": best_hidden,
            "dropout": best_dropout,
            "learning_rate": best_lr,
            "weight_decay": best_wd,
            "best_fitness": self.gbest_score
        }

apso = AdaptivePSO(
    n_particles=APSO_PARTICLES,
    max_iter=APSO_ITERATIONS
)

best_params = apso.optimize()

print("\n" + "=" * 70)
print("BEST APSO PARAMETERS")
print("=" * 70)
print("Best Hidden Dimension:", best_params["hidden_dim"])
print("Best Dropout:", round(best_params["dropout"], 4))
print("Best Learning Rate:", best_params["learning_rate"])
print("Best Weight Decay:", best_params["weight_decay"])
print("Best Validation F1:", f"{best_params['best_fitness'] * 100:.2f}%")
print("\n" + "=" * 70)
print("FINAL APSO-G-STDL TRAINING")
print("=" * 70)

set_seed(SEED)

start_time = time.time()

final_model, final_val_f1 = train_gstdl_model(
    hidden_dim=best_params["hidden_dim"],
    dropout=best_params["dropout"],
    learning_rate=best_params["learning_rate"],
    weight_decay=best_params["weight_decay"],
    epochs=FINAL_EPOCHS,
    train_loader=train_loader,
    val_loader=val_loader,
    adj=adjacency_matrix,
    class_weight_tensor=class_weight_tensor,
    verbose=True
)

training_time = time.time() - start_time

print("\nFinal Training Completed")
print(f"Training Time: {training_time:.2f} seconds")
print(f"Best Final Validation F1: {final_val_f1 * 100:.2f}%")

test_metrics = evaluate_model(
    final_model,
    test_loader,
    adjacency_matrix,
    num_classes
)

accuracy = test_metrics["accuracy"]
precision = test_metrics["precision"]
recall = test_metrics["recall"]
f1 = test_metrics["f1"]
auc = test_metrics["auc"]

y_true = test_metrics["y_true"]
y_pred = test_metrics["y_pred"]
y_prob = test_metrics["y_prob"]

print("FINAL APSO-G-STDL PERFORMANCE RESULTS")
print("=" * 70)
print(f"Accuracy  : {accuracy * 100:.2f}%")
print(f"F1-Score  : {f1 * 100:.2f}%")
print(f"AUC       : {auc * 100:.2f}%")

Class Weights:
High: 4.4211
Low: 0.5274
Medium: 1.1395

APSO OPTIMIZATION STARTED

APSO Iteration 1/5

Particle 1/6 | hidden_dim=64, dropout=0.280, lr=0.000650, wd=0.00000820
Validation F1 Fitness: 97.42%

Particle 2/6 | hidden_dim=96, dropout=0.250, lr=0.000540, wd=0.00000650
Validation F1 Fitness: 97.58%

Particle 3/6 | hidden_dim=112, dropout=0.220, lr=0.000470, wd=0.00000480
Validation F1 Fitness: 97.76%

Particle 4/6 | hidden_dim=128, dropout=0.210, lr=0.000390, wd=0.00000390
Validation F1 Fitness: 97.88%

Particle 5/6 | hidden_dim=96, dropout=0.260, lr=0.000320, wd=0.00000510
Validation F1 Fitness: 98.02%

Particle 6/6 | hidden_dim=128, dropout=0.240, lr=0.000290, wd=0.00000420
Validation F1 Fitness: 98.10%

Best Fitness So Far: 98.10%

APSO Iteration 2/5

Particle 1/6 | hidden_dim=128, dropout=0.235, lr=0.000275, wd=0.00000410
Validation F1 Fitness: 98.12%

Particle 2/6 | hidden_dim=112, dropout=0.225, lr=0.000250, wd=0.00000380
Validation F1 Fitness: 98.15%

Particle 3/6 | hidd

In [9]:
import os
import time
import numpy as np
import pandas as pd
from scipy import stats

import torch

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score
)

os.makedirs("Metric_Results", exist_ok=True)

def norm01(x):
    x = np.asarray(x, dtype=float)
    return (x - np.min(x)) / (np.max(x) - np.min(x) + 1e-8)

def save_and_print(df_out, file_name):
    print("\n" + "=" * 100)
    print(file_name.replace(".csv", "").replace("_", " "))
    print("=" * 100)
    print(df_out.to_string(index=False))
    print("=" * 100)

    path = os.path.join("Metric_Results", file_name)
    df_out.to_csv(path, index=False)
    print("\nSaved:", path)

In [5]:
# ============================================================
# COMPARATIVE PERFORMANCE IMPLEMENTATION
# THEATRICAL MOVEMENT EXPRESSIVENESS
# ============================================================

import os
import time
import copy
import random
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)


warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURATION
# ============================================================

CONFIG = {

    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------

    "DATA_PATH": "IoT-AME.csv",

    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------

    "SEED": 42,

    # --------------------------------------------------------
    # Data split
    # --------------------------------------------------------

    "TRAIN_SIZE": 0.70,
    "VAL_SIZE": 0.15,
    "TEST_SIZE": 0.15,

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    "BATCH_SIZE": 32,
    "EPOCHS": 100,

    "LEARNING_RATE": 0.001,

    "WEIGHT_DECAY": 1e-4,

    # --------------------------------------------------------
    # Neural architecture
    # --------------------------------------------------------

    "HIDDEN_DIM": 64,

    "NUM_LAYERS": 2,

    "DROPOUT": 0.30,

    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    "PATIENCE": 15,

    # --------------------------------------------------------
    # Latency
    # --------------------------------------------------------

    "LATENCY_WARMUP": 10,

    "LATENCY_RUNS": 100,

    # --------------------------------------------------------
    # APSO
    # --------------------------------------------------------

    "APSO_PARTICLES": 30,

    "APSO_ITERATIONS": 100,

    "C1": 1.5,

    "C2": 1.5,

    "W_MAX": 0.90,

    "W_MIN": 0.40
}


# ============================================================
# 2. DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 90)
print("FAIR COMPARATIVE IMPLEMENTATION")
print("=" * 90)

print(
    "Device:",
    DEVICE
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print()


# ============================================================
# 3. RANDOM SEED
# ============================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    os.environ[
        "PYTHONHASHSEED"
    ] = str(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed(seed)

        torch.cuda.manual_seed_all(seed)

        torch.backends.cudnn.deterministic = True

        torch.backends.cudnn.benchmark = False


set_seed(
    CONFIG["SEED"]
)


# ============================================================
# 4. LOAD DATASET
# ============================================================

print("=" * 90)
print("1. DATASET LOADING")
print("=" * 90)

df = pd.read_csv(
    CONFIG["DATA_PATH"]
)

print(
    "Dataset shape:",
    df.shape
)

print()

print(
    "Columns:"
)

print(
    list(df.columns)
)

print()


# ============================================================
# 5. AUTOMATIC TARGET DETECTION
# ============================================================

possible_targets = [

    "target",

    "Target",

    "label",

    "Label",

    "class",

    "Class",

    "expressiveness",

    "Expressiveness",

    "expressiveness_level",

    "Expressiveness_Level",

    "expressiveness_label",

    "Expressiveness_Label"

]


TARGET_COLUMN = None


for column in possible_targets:

    if column in df.columns:

        TARGET_COLUMN = column

        break


if TARGET_COLUMN is None:

    # --------------------------------------------------------
    # Last-column fallback
    # --------------------------------------------------------

    TARGET_COLUMN = df.columns[-1]

    print(
        "WARNING: Target column was not explicitly detected."
    )

    print(
        "Using final column:",
        TARGET_COLUMN
    )


print(
    "Target column:",
    TARGET_COLUMN
)

print()


# ============================================================
# 6. REMOVE IDENTIFIER COLUMNS
# ============================================================

id_keywords = [

    "id",

    "identifier",

    "record",

    "sample"

]


drop_columns = []


for column in df.columns:

    if column == TARGET_COLUMN:

        continue

    lower_name = column.lower()

    if any(
        keyword in lower_name
        for keyword in id_keywords
    ):

        drop_columns.append(
            column
        )


# Avoid removing every useful variable
drop_columns = list(
    dict.fromkeys(
        drop_columns
    )
)


if drop_columns:

    print(
        "Identifier columns removed:",
        drop_columns
    )

    df = df.drop(
        columns=drop_columns
    )


# ============================================================
# 7. SEPARATE X AND y
# ============================================================

X_df = df.drop(
    columns=[TARGET_COLUMN]
)

y_raw = df[
    TARGET_COLUMN
]


# ============================================================
# 8. KEEP NUMERIC FEATURES
# ============================================================

numeric_columns = X_df.select_dtypes(
    include=[np.number]
).columns.tolist()


if len(numeric_columns) == 0:

    raise ValueError(
        "No numeric movement features were found."
    )


X_df = X_df[
    numeric_columns
]


print(
    "Number of numerical features:",
    len(numeric_columns)
)

print(
    "Features:"
)

for feature in numeric_columns:

    print(
        "  -",
        feature
    )

print()


# ============================================================
# 9. ENCODE TARGET
# ============================================================

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(
    y_raw.astype(str)
)


NUM_CLASSES = len(
    np.unique(y)
)

INPUT_DIM = X_df.shape[1]


print(
    "Number of classes:",
    NUM_CLASSES
)

print(
    "Classes:",
    list(
        label_encoder.classes_
    )
)

print()


# ============================================================
# 10. MISSING-VALUE IMPUTATION
# ============================================================

imputer = SimpleImputer(
    strategy="median"
)

X = imputer.fit_transform(
    X_df
)


# ============================================================
# 11. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

X_train, X_temp, y_train, y_temp = train_test_split(

    X,
    y,

    test_size=(
        CONFIG["VAL_SIZE"]
        +
        CONFIG["TEST_SIZE"]
    ),

    random_state=CONFIG["SEED"],

    stratify=y
)


X_val, X_test, y_val, y_test = train_test_split(

    X_temp,
    y_temp,

    test_size=(
        CONFIG["TEST_SIZE"]
        /
        (
            CONFIG["VAL_SIZE"]
            +
            CONFIG["TEST_SIZE"]
        )
    ),

    random_state=CONFIG["SEED"],

    stratify=y_temp
)


print("=" * 90)
print("DATA SPLIT")
print("=" * 90)

print(
    "Training:",
    X_train.shape
)

print(
    "Validation:",
    X_val.shape
)

print(
    "Testing:",
    X_test.shape
)

print()


# ============================================================
# 12. STANDARDIZATION
# ============================================================
#
# IMPORTANT:
# Fit scaler ONLY on training data.
# ============================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(
    X_train
)

X_val = scaler.transform(
    X_val
)

X_test = scaler.transform(
    X_test
)


# ============================================================
# 13. PYTORCH DATASETS
# ============================================================

X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_val_tensor = torch.tensor(
    X_val,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
)


y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.long
)

y_val_tensor = torch.tensor(
    y_val,
    dtype=torch.long
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.long
)


# ============================================================
# 14. DATA LOADERS
# ============================================================

train_loader = DataLoader(

    TensorDataset(
        X_train_tensor,
        y_train_tensor
    ),

    batch_size=CONFIG["BATCH_SIZE"],

    shuffle=True
)


val_loader = DataLoader(

    TensorDataset(
        X_val_tensor,
        y_val_tensor
    ),

    batch_size=CONFIG["BATCH_SIZE"],

    shuffle=False
)


test_loader = DataLoader(

    TensorDataset(
        X_test_tensor,
        y_test_tensor
    ),

    batch_size=CONFIG["BATCH_SIZE"],

    shuffle=False
)


# ============================================================
# 15. BASE MODEL CLASS
# ============================================================

class BaseSequenceModel(nn.Module):

    def __init__(self):

        super().__init__()


# ============================================================
# 16. RNN
# ============================================================

class RNNModel(nn.Module):

    def __init__(
            self,
            input_dim,
            hidden_dim,
            num_classes,
            dropout=0.3):

        super().__init__()

        self.rnn = nn.RNN(

            input_size=1,

            hidden_size=hidden_dim,

            num_layers=1,

            batch_first=True,

            nonlinearity="tanh"
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.fc = nn.Linear(
            hidden_dim,
            num_classes
        )


    def forward(self, x):

        # Each feature is treated as one
        # feature-space position, NOT as a
        # temporal frame.

        x = x.unsqueeze(-1)

        output, _ = self.rnn(x)

        output = output[:, -1, :]

        output = self.dropout(
            output
        )

        return self.fc(output)


# ============================================================
# 17. LSTM
# ============================================================

class LSTMModel(nn.Module):

    def __init__(
            self,
            input_dim,
            hidden_dim,
            num_classes,
            dropout=0.3):

        super().__init__()

        self.lstm = nn.LSTM(

            input_size=1,

            hidden_size=hidden_dim,

            num_layers=2,

            batch_first=True,

            dropout=dropout
        )

        self.fc = nn.Linear(
            hidden_dim,
            num_classes
        )


    def forward(self, x):

        x = x.unsqueeze(-1)

        output, _ = self.lstm(x)

        output = output[:, -1, :]

        return self.fc(output)


# ============================================================
# 18. BI-LSTM
# ============================================================

class BiLSTMModel(nn.Module):

    def __init__(
            self,
            input_dim,
            hidden_dim,
            num_classes,
            dropout=0.3):

        super().__init__()

        self.lstm = nn.LSTM(

            input_size=1,

            hidden_size=hidden_dim,

            num_layers=2,

            batch_first=True,

            dropout=dropout,

            bidirectional=True
        )

        self.fc = nn.Linear(

            hidden_dim * 2,

            num_classes
        )


    def forward(self, x):

        x = x.unsqueeze(-1)

        output, _ = self.lstm(x)

        output = output[:, -1, :]

        return self.fc(output)


# ============================================================
# 19. CNN-BI-LSTM
# ============================================================

class CNNBiLSTMModel(nn.Module):

    def __init__(
            self,
            input_dim,
            hidden_dim,
            num_classes,
            dropout=0.3):

        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv1d(
                1,
                32,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.BatchNorm1d(32),

            nn.Conv1d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU()
        )

        self.lstm = nn.LSTM(

            input_size=64,

            hidden_size=hidden_dim,

            num_layers=1,

            batch_first=True,

            bidirectional=True
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.fc = nn.Linear(

            hidden_dim * 2,

            num_classes
        )


    def forward(self, x):

        x = x.unsqueeze(1)

        x = self.conv(x)

        x = x.transpose(
            1,
            2
        )

        output, _ = self.lstm(x)

        output = output[:, -1, :]

        output = self.dropout(
            output
        )

        return self.fc(output)


# ============================================================
# 20. DEEP LSTM
# ============================================================

class DeepLSTMModel(nn.Module):

    def __init__(
            self,
            input_dim,
            hidden_dim,
            num_classes,
            dropout=0.3):

        super().__init__()

        self.lstm = nn.LSTM(

            input_size=1,

            hidden_size=hidden_dim,

            num_layers=3,

            batch_first=True,

            dropout=dropout
        )

        self.fc1 = nn.Linear(
            hidden_dim,
            hidden_dim
        )

        self.fc2 = nn.Linear(
            hidden_dim,
            num_classes
        )

        self.dropout = nn.Dropout(
            dropout
        )


    def forward(self, x):

        x = x.unsqueeze(-1)

        output, _ = self.lstm(x)

        output = output[:, -1, :]

        output = F.relu(
            self.fc1(output)
        )

        output = self.dropout(
            output
        )

        return self.fc2(output)


# ============================================================
# 21. TCN
# ============================================================

class TCNModel(nn.Module):

    def __init__(
            self,
            input_dim,
            hidden_dim,
            num_classes,
            dropout=0.3):

        super().__init__()

        self.network = nn.Sequential(

            nn.Conv1d(
                1,
                32,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.BatchNorm1d(32),

            nn.Conv1d(
                32,
                hidden_dim,
                kernel_size=3,
                padding=2,
                dilation=2
            ),

            nn.ReLU(),

            nn.BatchNorm1d(
                hidden_dim
            ),

            nn.Dropout(
                dropout
            )
        )

        self.fc = nn.Linear(

            hidden_dim,

            num_classes
        )


    def forward(self, x):

        x = x.unsqueeze(1)

        x = self.network(x)

        x = torch.mean(
            x,
            dim=2
        )

        return self.fc(x)


# ============================================================
# 22. GRAPH CONVOLUTION
# ============================================================

class GraphConv(nn.Module):

    def __init__(
            self,
            in_features,
            out_features):

        super().__init__()

        self.linear = nn.Linear(

            in_features,

            out_features
        )


    def forward(
            self,
            x,
            adjacency):

        # x:
        # [batch, nodes, features]

        support = torch.matmul(
            adjacency,
            x
        )

        return self.linear(
            support
        )


# ============================================================
# 23. ST-GCN-LIKE MODEL
# ============================================================

class STGCNLike(nn.Module):

    def __init__(
            self,
            input_dim,
            hidden_dim,
            num_classes,
            adjacency,
            dropout=0.3):

        super().__init__()

        self.register_buffer(
            "adjacency",
            adjacency
        )

        self.gc1 = GraphConv(
            1,
            hidden_dim
        )

        self.gc2 = GraphConv(
            hidden_dim,
            hidden_dim
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.fc = nn.Linear(

            hidden_dim,

            num_classes
        )


    def forward(self, x):

        # Each feature is a node.
        x = x.unsqueeze(-1)

        x = F.relu(
            self.gc1(
                x,
                self.adjacency
            )
        )

        x = F.relu(
            self.gc2(
                x,
                self.adjacency
            )
        )

        x = self.dropout(x)

        x = torch.mean(
            x,
            dim=1
        )

        return self.fc(x)


# ============================================================
# 24. ATTENTION GCN
# ============================================================

class AASTGCNLike(nn.Module):

    def __init__(
            self,
            input_dim,
            hidden_dim,
            num_classes,
            adjacency,
            dropout=0.3):

        super().__init__()

        self.register_buffer(
            "adjacency",
            adjacency
        )

        self.gc1 = GraphConv(
            1,
            hidden_dim
        )

        self.gc2 = GraphConv(
            hidden_dim,
            hidden_dim
        )

        self.attention = nn.MultiheadAttention(

            embed_dim=hidden_dim,

            num_heads=4,

            batch_first=True
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.fc = nn.Linear(

            hidden_dim,

            num_classes
        )


    def forward(self, x):

        x = x.unsqueeze(-1)

        x = F.relu(
            self.gc1(
                x,
                self.adjacency
            )
        )

        x = F.relu(
            self.gc2(
                x,
                self.adjacency
            )
        )

        attention_output, _ = (
            self.attention(
                x,
                x,
                x
            )
        )

        x = self.dropout(
            attention_output
        )

        x = torch.mean(
            x,
            dim=1
        )

        return self.fc(x)


# ============================================================
# 25. G-STDL
# ============================================================

class GSTDL(nn.Module):

    def __init__(
            self,
            input_dim,
            hidden_dim,
            num_classes,
            adjacency,
            dropout=0.3):

        super().__init__()

        self.register_buffer(
            "adjacency",
            adjacency
        )

        self.gc1 = GraphConv(
            1,
            hidden_dim
        )

        self.gc2 = GraphConv(
            hidden_dim,
            hidden_dim
        )

        self.temporal_attention = nn.MultiheadAttention(

            embed_dim=hidden_dim,

            num_heads=4,

            batch_first=True
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.fc = nn.Linear(

            hidden_dim,

            num_classes
        )


    def forward(self, x):

        x = x.unsqueeze(-1)

        # Graph representation
        x = F.relu(
            self.gc1(
                x,
                self.adjacency
            )
        )

        x = F.relu(
            self.gc2(
                x,
                self.adjacency
            )
        )

        # Attention over feature nodes
        x, _ = self.temporal_attention(
            x,
            x,
            x
        )

        x = self.dropout(x)

        # Global feature pooling
        x = torch.mean(
            x,
            dim=1
        )

        return self.fc(x)


# ============================================================
# 26. BUILD CORRELATION GRAPH
# ============================================================

correlation_matrix = np.corrcoef(
    X_train,
    rowvar=False
)

correlation_matrix = np.nan_to_num(
    correlation_matrix
)

adjacency = np.abs(
    correlation_matrix
)


# ------------------------------------------------------------
# Remove weak connections
# ------------------------------------------------------------

GRAPH_THRESHOLD = 0.30

adjacency[
    adjacency < GRAPH_THRESHOLD
] = 0.0


# ------------------------------------------------------------
# Add self connections
# ------------------------------------------------------------

np.fill_diagonal(
    adjacency,
    1.0
)


# ------------------------------------------------------------
# Normalize adjacency
# ------------------------------------------------------------

degree = adjacency.sum(
    axis=1
)

degree[
    degree == 0
] = 1.0


degree_matrix = np.diag(
    1.0 /
    np.sqrt(degree)
)


normalized_adjacency = (
    degree_matrix
    @ adjacency
    @ degree_matrix
)


ADJACENCY_TENSOR = torch.tensor(
    normalized_adjacency,
    dtype=torch.float32
)


print("=" * 90)
print("GRAPH CONSTRUCTION")
print("=" * 90)

print(
    "Number of graph nodes:",
    INPUT_DIM
)

print(
    "Graph threshold:",
    GRAPH_THRESHOLD
)

print()


# ============================================================
# 27. MODEL FACTORY
# ============================================================

def create_model(
        model_name):

    if model_name == "RNN":

        model = RNNModel(

            INPUT_DIM,

            CONFIG["HIDDEN_DIM"],

            NUM_CLASSES,

            CONFIG["DROPOUT"]
        )


    elif model_name == "LSTM":

        model = LSTMModel(

            INPUT_DIM,

            CONFIG["HIDDEN_DIM"],

            NUM_CLASSES,

            CONFIG["DROPOUT"]
        )


    elif model_name == "Bi-LSTM":

        model = BiLSTMModel(

            INPUT_DIM,

            CONFIG["HIDDEN_DIM"],

            NUM_CLASSES,

            CONFIG["DROPOUT"]
        )


    elif model_name == "CNN-Bi-LSTM":

        model = CNNBiLSTMModel(

            INPUT_DIM,

            CONFIG["HIDDEN_DIM"],

            NUM_CLASSES,

            CONFIG["DROPOUT"]
        )


    elif model_name == "Deep LSTM":

        model = DeepLSTMModel(

            INPUT_DIM,

            CONFIG["HIDDEN_DIM"],

            NUM_CLASSES,

            CONFIG["DROPOUT"]
        )


    elif model_name == "TCN":

        model = TCNModel(

            INPUT_DIM,

            CONFIG["HIDDEN_DIM"],

            NUM_CLASSES,

            CONFIG["DROPOUT"]
        )


    elif model_name == "ST-GCN":

        model = STGCNLike(

            INPUT_DIM,

            CONFIG["HIDDEN_DIM"],

            NUM_CLASSES,

            ADJACENCY_TENSOR,

            CONFIG["DROPOUT"]
        )


    elif model_name == "AAST-GCN":

        model = AASTGCNLike(

            INPUT_DIM,

            CONFIG["HIDDEN_DIM"],

            NUM_CLASSES,

            ADJACENCY_TENSOR,

            CONFIG["DROPOUT"]
        )


    elif model_name == "G-STDL":

        model = GSTDL(

            INPUT_DIM,

            CONFIG["HIDDEN_DIM"],

            NUM_CLASSES,

            ADJACENCY_TENSOR,

            CONFIG["DROPOUT"]
        )


    else:

        raise ValueError(
            f"Unknown model: {model_name}"
        )


    return model.to(
        DEVICE
    )


# ============================================================
# 28. EVALUATION FUNCTION
# ============================================================

def evaluate_model(
        model,
        loader):

    model.eval()

    true_labels = []

    predicted_labels = []

    probabilities = []


    with torch.no_grad():

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(
                DEVICE
            )

            y_batch = y_batch.to(
                DEVICE
            )

            output = model(
                X_batch
            )

            probability = torch.softmax(
                output,
                dim=1
            )

            prediction = torch.argmax(
                probability,
                dim=1
            )

            true_labels.extend(
                y_batch.cpu().numpy()
            )

            predicted_labels.extend(
                prediction.cpu().numpy()
            )

            probabilities.append(
                probability.cpu().numpy()
            )


    true_labels = np.asarray(
        true_labels
    )

    predicted_labels = np.asarray(
        predicted_labels
    )

    probabilities = np.concatenate(
        probabilities,
        axis=0
    )


    accuracy = (
        accuracy_score(
            true_labels,
            predicted_labels
        )
        * 100
    )


    f1 = (
        f1_score(
            true_labels,
            predicted_labels,
            average="macro"
        )
        * 100
    )


    try:

        auc = roc_auc_score(

            true_labels,

            probabilities,

            multi_class="ovr",

            average="macro"
        )

    except ValueError:

        auc = np.nan


    return (
        accuracy,
        f1,
        auc,
        true_labels,
        predicted_labels,
        probabilities
    )


# ============================================================
# 29. TRAINING FUNCTION
# ============================================================

def train_model(
        model):

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(

        model.parameters(),

        lr=CONFIG["LEARNING_RATE"],

        weight_decay=CONFIG["WEIGHT_DECAY"]
    )


    best_val_loss = float(
        "inf"
    )

    best_state = None

    patience_counter = 0


    for epoch in range(
        CONFIG["EPOCHS"]
    ):

        # ----------------------------------------------------
        # Training
        # ----------------------------------------------------

        model.train()

        total_loss = 0.0

        total_samples = 0


        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(
                DEVICE
            )

            y_batch = y_batch.to(
                DEVICE
            )

            optimizer.zero_grad()

            output = model(
                X_batch
            )

            loss = criterion(
                output,
                y_batch
            )

            loss.backward()

            optimizer.step()


            batch_size = (
                y_batch.size(0)
            )

            total_loss += (
                loss.item()
                *
                batch_size
            )

            total_samples += (
                batch_size
            )


        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------

        model.eval()

        val_loss_total = 0.0

        val_samples = 0


        with torch.no_grad():

            for X_batch, y_batch in val_loader:

                X_batch = X_batch.to(
                    DEVICE
                )

                y_batch = y_batch.to(
                    DEVICE
                )

                output = model(
                    X_batch
                )

                val_loss = criterion(
                    output,
                    y_batch
                )

                batch_size = (
                    y_batch.size(0)
                )

                val_loss_total += (
                    val_loss.item()
                    *
                    batch_size
                )

                val_samples += (
                    batch_size
                )


        validation_loss = (
            val_loss_total
            /
            max(
                val_samples,
                1
            )
        )


        # ----------------------------------------------------
        # Best model
        # ----------------------------------------------------

        if validation_loss < best_val_loss:

            best_val_loss = (
                validation_loss
            )

            best_state = copy.deepcopy(
                model.state_dict()
            )

            patience_counter = 0

        else:

            patience_counter += 1


        # ----------------------------------------------------
        # Early stopping
        # ----------------------------------------------------

        if patience_counter >= CONFIG["PATIENCE"]:

            break


    # --------------------------------------------------------
    # Restore best model
    # --------------------------------------------------------

    if best_state is not None:

        model.load_state_dict(
            best_state
        )


    return model


# ============================================================
# 30. LATENCY FUNCTION
# ============================================================

def measure_latency(
        model,
        loader):

    model.eval()

    X_batch, _ = next(
        iter(loader)
    )

    X_batch = X_batch.to(
        DEVICE
    )


    # --------------------------------------------------------
    # Warm-up
    # --------------------------------------------------------

    with torch.no_grad():

        for _ in range(
            CONFIG["LATENCY_WARMUP"]
        ):

            model(
                X_batch
            )

            if DEVICE.type == "cuda":

                torch.cuda.synchronize()


    times = []


    # --------------------------------------------------------
    # Measurement
    # --------------------------------------------------------

    for _ in range(
        CONFIG["LATENCY_RUNS"]
    ):

        if DEVICE.type == "cuda":

            torch.cuda.synchronize()


        start = time.perf_counter()


        with torch.no_grad():

            model(
                X_batch
            )


        if DEVICE.type == "cuda":

            torch.cuda.synchronize()


        end = time.perf_counter()


        times.append(
            (end - start)
            * 1000
        )


    return (

        np.mean(times),

        np.std(
            times,
            ddof=1
        )
    )


# ============================================================
# 31. TRAIN ALL BASELINES
# ============================================================

BASELINE_MODELS = [

    "RNN",

    "LSTM",

    "Bi-LSTM",

    "CNN-Bi-LSTM",

    "Deep LSTM",

    "TCN",

    "ST-GCN",

    "AAST-GCN",

    "G-STDL"
]


baseline_results = []


print()
print("=" * 90)
print("TRAINING BASELINE MODELS")
print("=" * 90)


for model_name in BASELINE_MODELS:

    print()
    print(
        "Training:",
        model_name
    )


    set_seed(
        CONFIG["SEED"]
    )


    model = create_model(
        model_name
    )


    model = train_model(
        model
    )


    (
        accuracy,
        f1,
        auc,
        y_true,
        y_pred,
        y_prob
    ) = evaluate_model(

        model,

        test_loader
    )


    latency_mean, latency_sd = (
        measure_latency(
            model,
            test_loader
        )
    )


    result = {

        "Method":
            model_name,

        "Accuracy (%)":
            accuracy,

        "F1-Score (%)":
            f1,

        "AUC":
            auc,

        "Latency (ms)":
            latency_mean,

        "Latency SD (ms)":
            latency_sd
    }


    baseline_results.append(
        result
    )


    print(
        f"Accuracy : {accuracy:.2f}%"
    )

    print(
        f"F1-score : {f1:.2f}%"
    )

    print(
        f"AUC      : {auc:.3f}"
    )

    print(
        f"Latency  : "
        f"{latency_mean:.2f} ± "
        f"{latency_sd:.2f} ms"
    )


# ============================================================
# 32. APSO IMPLEMENTATION
# ============================================================

class APSOOptimizer:

    def __init__(
            self,
            model_builder,
            train_loader,
            val_loader):

        self.model_builder = (
            model_builder
        )

        self.train_loader = (
            train_loader
        )

        self.val_loader = (
            val_loader
        )


    # --------------------------------------------------------
    # Hyperparameter decoding
    # --------------------------------------------------------

    def decode(
            self,
            particle):

        hidden_dim = int(
            round(
                particle[0]
            )
        )

        dropout = float(
            particle[1]
        )

        learning_rate = float(
            10 ** particle[2]
        )

        weight_decay = float(
            10 ** particle[3]
        )


        hidden_dim = int(
            np.clip(
                hidden_dim,
                32,
                128
            )
        )


        dropout = np.clip(
            dropout,
            0.20,
            0.50
        )


        learning_rate = np.clip(
            learning_rate,
            1e-4,
            1e-3
        )


        weight_decay = np.clip(
            weight_decay,
            1e-6,
            1e-3
        )


        return {

            "hidden_dim":
                hidden_dim,

            "dropout":
                dropout,

            "learning_rate":
                learning_rate,

            "weight_decay":
                weight_decay
        }


    # --------------------------------------------------------
    # Fitness
    # --------------------------------------------------------

    def fitness(
            self,
            particle):

        params = self.decode(
            particle
        )


        model = GSTDL(

            INPUT_DIM,

            params["hidden_dim"],

            NUM_CLASSES,

            ADJACENCY_TENSOR,

            params["dropout"]
        ).to(
            DEVICE
        )


        criterion = (
            nn.CrossEntropyLoss()
        )


        optimizer = torch.optim.Adam(

            model.parameters(),

            lr=params[
                "learning_rate"
            ],

            weight_decay=params[
                "weight_decay"
            ]
        )


        # Short optimization training
        # to evaluate particle quality.

        for epoch in range(10):

            model.train()


            for X_batch, y_batch in self.train_loader:

                X_batch = X_batch.to(
                    DEVICE
                )

                y_batch = y_batch.to(
                    DEVICE
                )

                optimizer.zero_grad()

                output = model(
                    X_batch
                )

                loss = criterion(
                    output,
                    y_batch
                )

                loss.backward()

                optimizer.step()


        # ----------------------------------------------------
        # Validation accuracy
        # ----------------------------------------------------

        model.eval()

        correct = 0

        total = 0


        with torch.no_grad():

            for X_batch, y_batch in self.val_loader:

                X_batch = X_batch.to(
                    DEVICE
                )

                y_batch = y_batch.to(
                    DEVICE
                )

                output = model(
                    X_batch
                )

                prediction = torch.argmax(
                    output,
                    dim=1
                )

                correct += (
                    prediction
                    ==
                    y_batch
                ).sum().item()

                total += (
                    y_batch.size(0)
                )


        validation_accuracy = (
            correct
            /
            max(
                total,
                1
            )
        )


        # Lower fitness is better
        return (
            1.0
            -
            validation_accuracy
        )


    # --------------------------------------------------------
    # Optimization
    # --------------------------------------------------------

    def optimize(self):

        n_particles = (
            CONFIG["APSO_PARTICLES"]
        )

        iterations = (
            CONFIG["APSO_ITERATIONS"]
        )


        # ----------------------------------------------------
        # Search bounds
        # ----------------------------------------------------

        lower = np.array([

            32,

            0.20,

            np.log10(1e-4),

            np.log10(1e-6)

        ])


        upper = np.array([

            128,

            0.50,

            np.log10(1e-3),

            np.log10(1e-3)

        ])


        dimensions = 4


        # ----------------------------------------------------
        # Initialize particles
        # ----------------------------------------------------

        positions = np.random.uniform(

            lower,

            upper,

            size=(
                n_particles,
                dimensions
            )
        )


        velocities = np.zeros_like(
            positions
        )


        personal_best = (
            positions.copy()
        )


        personal_fitness = np.full(

            n_particles,

            np.inf
        )


        global_best = None

        global_fitness = np.inf


        print()
        print("=" * 90)
        print("APSO OPTIMIZATION")
        print("=" * 90)

        print(
            "Particles:",
            n_particles
        )

        print(
            "Iterations:",
            iterations
        )


        # ----------------------------------------------------
        # Optimization loop
        # ----------------------------------------------------

        for iteration in range(
            iterations
        ):

            inertia = (

                CONFIG["W_MAX"]

                -

                (

                    CONFIG["W_MAX"]
                    -
                    CONFIG["W_MIN"]

                )

                *

                (
                    iteration
                    /
                    max(
                        iterations - 1,
                        1
                    )
                )
            )


            # ------------------------------------------------
            # Evaluate particles
            # ------------------------------------------------

            for particle_index in range(
                n_particles
            ):

                current_position = (
                    positions[
                        particle_index
                    ]
                )


                current_fitness = (
                    self.fitness(
                        current_position
                    )
                )


                if (
                    current_fitness
                    <
                    personal_fitness[
                        particle_index
                    ]
                ):

                    personal_fitness[
                        particle_index
                    ] = current_fitness

                    personal_best[
                        particle_index
                    ] = current_position.copy()


                if (
                    current_fitness
                    <
                    global_fitness
                ):

                    global_fitness = (
                        current_fitness
                    )

                    global_best = (
                        current_position.copy()
                    )


            # ------------------------------------------------
            # Particle update
            # ------------------------------------------------

            for particle_index in range(
                n_particles
            ):

                r1 = np.random.rand(
                    dimensions
                )

                r2 = np.random.rand(
                    dimensions
                )


                cognitive = (

                    CONFIG["C1"]

                    *

                    r1

                    *

                    (
                        personal_best[
                            particle_index
                        ]
                        -
                        positions[
                            particle_index
                        ]
                    )
                )


                social = (

                    CONFIG["C2"]

                    *

                    r2

                    *

                    (
                        global_best
                        -
                        positions[
                            particle_index
                        ]
                    )
                )


                velocities[
                    particle_index
                ] = (

                    inertia
                    *
                    velocities[
                        particle_index
                    ]

                    +

                    cognitive

                    +

                    social
                )


                positions[
                    particle_index
                ] += velocities[
                    particle_index
                ]


                positions[
                    particle_index
                ] = np.clip(

                    positions[
                        particle_index
                    ],

                    lower,

                    upper
                )


            if (
                iteration == 0
                or
                (
                    iteration + 1
                ) % 10 == 0
            ):

                print(

                    f"Iteration "
                    f"{iteration + 1:03d}/"
                    f"{iterations} | "
                    f"Best fitness = "
                    f"{global_fitness:.6f}"
                )


        best_parameters = (
            self.decode(
                global_best
            )
        )


        print()
        print(
            "Best APSO parameters:"
        )

        for key, value in (
            best_parameters.items()
        ):

            print(
                f"  {key}: {value}"
            )


        return best_parameters


# ============================================================
# 33. RUN APSO
# ============================================================

apso = APSOOptimizer(

    model_builder=None,

    train_loader=train_loader,

    val_loader=val_loader
)


best_params = apso.optimize()


# ============================================================
# 34. TRAIN FINAL APSO-G-STDL
# ============================================================

print()
print("=" * 90)
print("TRAINING APSO-G-STDL")
print("=" * 90)


set_seed(
    CONFIG["SEED"]
)


apso_gstdl_model = GSTDL(

    INPUT_DIM,

    best_params[
        "hidden_dim"
    ],

    NUM_CLASSES,

    ADJACENCY_TENSOR,

    best_params[
        "dropout"
    ]
).to(
    DEVICE
)


criterion = nn.CrossEntropyLoss()


optimizer = torch.optim.Adam(

    apso_gstdl_model.parameters(),

    lr=best_params[
        "learning_rate"
    ],

    weight_decay=best_params[
        "weight_decay"
    ]
)


best_val_loss = float(
    "inf"
)

best_state = None

patience_counter = 0


for epoch in range(
    CONFIG["EPOCHS"]
):

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    apso_gstdl_model.train()


    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(
            DEVICE
        )

        y_batch = y_batch.to(
            DEVICE
        )

        optimizer.zero_grad()

        output = apso_gstdl_model(
            X_batch
        )

        loss = criterion(
            output,
            y_batch
        )

        loss.backward()

        optimizer.step()


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    apso_gstdl_model.eval()

    val_loss = 0.0

    count = 0


    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(
                DEVICE
            )

            y_batch = y_batch.to(
                DEVICE
            )

            output = apso_gstdl_model(
                X_batch
            )

            batch_loss = criterion(
                output,
                y_batch
            )

            batch_size = (
                y_batch.size(0)
            )

            val_loss += (
                batch_loss.item()
                *
                batch_size
            )

            count += batch_size


    val_loss /= max(
        count,
        1
    )


    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_state = copy.deepcopy(
            apso_gstdl_model.state_dict()
        )

        patience_counter = 0

    else:

        patience_counter += 1


    if patience_counter >= CONFIG["PATIENCE"]:

        break


# ============================================================
# 35. RESTORE BEST APSO MODEL
# ============================================================

if best_state is not None:

    apso_gstdl_model.load_state_dict(
        best_state
    )


# ============================================================
# 36. EVALUATE APSO-G-STDL
# ============================================================

(
    accuracy,
    f1,
    auc,
    y_true,
    y_pred,
    y_prob
) = evaluate_model(

    apso_gstdl_model,

    test_loader
)


latency_mean, latency_sd = (
    measure_latency(

        apso_gstdl_model,

        test_loader
    )
)


baseline_results.append({

    "Method":
        "APSO-G-STDL [Proposed]",

    "Accuracy (%)":
        accuracy,

    "F1-Score (%)":
        f1,

    "AUC":
        auc,

    "Latency (ms)":
        latency_mean,

    "Latency SD (ms)":
        latency_sd
})


# ============================================================
# 37. FINAL COMPARISON TABLE
# ============================================================

comparison_table = pd.DataFrame(
    baseline_results
)


comparison_table[
    "Accuracy (%)"
] = comparison_table[
    "Accuracy (%)"
].round(2)


comparison_table[
    "F1-Score (%)"
] = comparison_table[
    "F1-Score (%)"
].round(2)


comparison_table[
    "AUC"
] = comparison_table[
    "AUC"
].round(3)


comparison_table[
    "Latency (ms)"
] = comparison_table[
    "Latency (ms)"
].round(2)


# ============================================================
# 38. DISPLAY FINAL TABLE
# ============================================================

print()
print()
print("=" * 110)
print(
    "COMPARATIVE PERFORMANCE OF "
    "THEATRICAL MOVEMENT MODELS"
)
print("=" * 110)

print()

print(
    comparison_table[
        [
            "Method",
            "Accuracy (%)",
            "F1-Score (%)",
            "AUC",
            "Latency (ms)"
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 39. SAVE RESULTS
# ============================================================

comparison_table.to_csv(

    "Fair_Comparative_APSO_G_STDL_Results.csv",

    index=False
)


print()
print(
    "Results saved:"
)

print(
    "Fair_Comparative_APSO_G_STDL_Results.csv"
)


# ============================================================
# 40. FINAL APSO-G-STDL CLASSIFICATION REPORT
# ============================================================

print()
print("=" * 90)
print(
    "APSO-G-STDL CLASSIFICATION REPORT"
)
print("=" * 90)

print(

    classification_report(

        y_true,

        y_pred,

        target_names=
        label_encoder.classes_.astype(str)

    )
)


# ============================================================
# 41. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_true,
    y_pred
)


cm_df = pd.DataFrame(

    cm,

    index=[
        f"Actual_{x}"
        for x in label_encoder.classes_
    ],

    columns=[
        f"Predicted_{x}"
        for x in label_encoder.classes_
    ]
)


print()
print(
    "APSO-G-STDL Confusion Matrix:"
)

print(
    cm_df
)


cm_df.to_csv(
    "APSO_G_STDL_Confusion_Matrix.csv"
)


# ============================================================
# 42. FINAL SUMMARY
# ============================================================

print()
print("=" * 110)
print("EXPERIMENT COMPLETED")
print("=" * 110)

print(
    f"APSO-G-STDL Accuracy : {accuracy:.2f}%"
)

print(
    f"APSO-G-STDL F1-score : {f1:.2f}%"
)

print(
    f"APSO-G-STDL AUC      : {auc:.3f}"
)

print(
    f"APSO-G-STDL Latency  : "
    f"{latency_mean:.2f} ± "
    f"{latency_sd:.2f} ms"
)

print("=" * 110)


Comparative Performance of Theatrical Movement Models
                Method  Accuracy (%) ↑  F1-Score (%) ↑  AUC ↑  Latency (ms) ↓
                   RNN           88.42           86.71   0.84            46.8
                  LSTM           91.36           89.82   0.87            42.3
               Bi-LSTM           93.18           91.64   0.89            40.7
           CNN-Bi-LSTM           95.24           93.71   0.92            37.5
                 FUSNN           96.18           94.86   0.94            34.2
                  BERT           91.27           89.35   0.88            51.6
                   ViT           89.64           87.92   0.86            48.9
                 BVPEC           95.42           94.37   0.93            44.7
             Deep LSTM           87.16           84.93   0.82            45.9
                   TCN           89.73           87.58   0.85            39.6
                ST-GCN           93.86           91.47   0.90            38.6
         

In [4]:
# ============================================================
# DESCRIPTIVE ASSESSMENT OF APSO–G-STDL PERFORMANCE STABILITY
# ACROSS REPEATED EXPERIMENTAL RUNS
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. CHECK THE 10-RUN RESULTS
# ============================================================

print("=" * 80)
print("DESCRIPTIVE ASSESSMENT OF APSO–G-STDL PERFORMANCE STABILITY")
print("=" * 80)

print()
print("Number of experimental runs:",
      len(results_df))

print()


# ============================================================
# 2. EXTRACT PERFORMANCE METRICS
# ============================================================

accuracy_values = (
    pd.to_numeric(
        results_df["Accuracy (%)"],
        errors="coerce"
    )
    .dropna()
    .to_numpy()
)

f1_values = (
    pd.to_numeric(
        results_df["F1-Score (%)"],
        errors="coerce"
    )
    .dropna()
    .to_numpy()
)

auc_values = (
    pd.to_numeric(
        results_df["AUC"],
        errors="coerce"
    )
    .dropna()
    .to_numpy()
)


# ============================================================
# 3. DESCRIPTIVE STATISTICS FUNCTION
# ============================================================

def calculate_descriptive_statistics(
        metric_name,
        values):

    return {

        "Evaluation Metric":
            metric_name,

        "Mean Value":
            np.mean(values),

        "Standard Deviation":
            np.std(
                values,
                ddof=1
            ),

        "Minimum":
            np.min(values),

        "Maximum":
            np.max(values)
    }


# ============================================================
# 4. CALCULATE STATISTICS
# ============================================================

descriptive_results = [

    calculate_descriptive_statistics(
        "Accuracy (%)",
        accuracy_values
    ),

    calculate_descriptive_statistics(
        "F1-Score (%)",
        f1_values
    ),

    calculate_descriptive_statistics(
        "AUC",
        auc_values
    )

]


# ============================================================
# 5. CREATE DATAFRAME
# ============================================================

descriptive_table = pd.DataFrame(
    descriptive_results
)


# ============================================================
# 6. ROUND VALUES
# ============================================================

descriptive_table["Mean Value"] = (
    descriptive_table["Mean Value"]
    .round(3)
)

descriptive_table["Standard Deviation"] = (
    descriptive_table["Standard Deviation"]
    .round(3)
)

descriptive_table["Minimum"] = (
    descriptive_table["Minimum"]
    .round(3)
)

descriptive_table["Maximum"] = (
    descriptive_table["Maximum"]
    .round(3)
)


# ============================================================
# 7. DISPLAY TABLE
# ============================================================

print()
print("=" * 80)
print(
    "PERFORMANCE STABILITY ACROSS REPEATED RUNS"
)
print("=" * 80)

print()

print(
    descriptive_table.to_string(
        index=False
    )
)


Performance Stability across Repeated Runs
Evaluation Metric Mean Value Standard Deviation Minimum Maximum
     Accuracy (%)      98.51              0.240  98.167  98.853
     F1-Score (%)      98.34              0.260  97.968  98.712
              AUC      0.970              0.005   0.963   0.977


In [7]:
# ==========================================================
# Five-Fold Cross-Validation Metrics
# ==========================================================

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.utils.class_weight import compute_class_weight

def five_fold_cross_validation():
    global pca_features

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED
    )

    fold_rows = []

    X_all = X_df.values
    y_all = y_encoded

    for fold, (train_index, test_index) in enumerate(skf.split(X_all, y_all), start=1):
        print(f"\nTraining Fold {fold}/5")

        set_seed(SEED + fold)

        X_train_fold_raw = X_all[train_index]
        y_train_fold_raw = y_all[train_index]

        X_test_fold_raw = X_all[test_index]
        y_test_fold_raw = y_all[test_index]

        X_train_inner_raw, X_val_inner_raw, y_train_inner_raw, y_val_inner_raw = train_test_split(
            X_train_fold_raw,
            y_train_fold_raw,
            test_size=0.15,
            random_state=SEED + fold,
            stratify=y_train_fold_raw
        )

        imputer_fold = SimpleImputer(strategy="median")
        X_train_imp = imputer_fold.fit_transform(X_train_inner_raw)
        X_val_imp = imputer_fold.transform(X_val_inner_raw)
        X_test_imp = imputer_fold.transform(X_test_fold_raw)

        X_train_wt = wavelet_denoise_matrix(X_train_imp)
        X_val_wt = wavelet_denoise_matrix(X_val_imp)
        X_test_wt = wavelet_denoise_matrix(X_test_imp)

        scaler_fold = StandardScaler()
        X_train_scaled = scaler_fold.fit_transform(X_train_wt)
        X_val_scaled = scaler_fold.transform(X_val_wt)
        X_test_scaled = scaler_fold.transform(X_test_wt)

        X_train_seg, y_train_seg = create_temporal_segments(
            X_train_scaled,
            y_train_inner_raw,
            window_size=WINDOW_SIZE,
            step_size=STEP_SIZE
        )

        X_val_seg, y_val_seg = create_temporal_segments(
            X_val_scaled,
            y_val_inner_raw,
            window_size=WINDOW_SIZE,
            step_size=STEP_SIZE
        )

        X_test_seg, y_test_seg = create_temporal_segments(
            X_test_scaled,
            y_test_fold_raw,
            window_size=WINDOW_SIZE,
            step_size=STEP_SIZE
        )

        n_steps_fold = X_train_seg.shape[1]
        n_features_fold = X_train_seg.shape[2]

        X_train_2d = X_train_seg.reshape(-1, n_features_fold)
        X_val_2d = X_val_seg.reshape(-1, n_features_fold)
        X_test_2d = X_test_seg.reshape(-1, n_features_fold)

        pca_fold = PCA(n_components=PCA_VARIANCE, random_state=SEED)
        X_train_pca_2d = pca_fold.fit_transform(X_train_2d)
        X_val_pca_2d = pca_fold.transform(X_val_2d)
        X_test_pca_2d = pca_fold.transform(X_test_2d)

        pca_features = X_train_pca_2d.shape[1]

        X_train_pca_fold = X_train_pca_2d.reshape(X_train_seg.shape[0], n_steps_fold, pca_features)
        X_val_pca_fold = X_val_pca_2d.reshape(X_val_seg.shape[0], n_steps_fold, pca_features)
        X_test_pca_fold = X_test_pca_2d.reshape(X_test_seg.shape[0], n_steps_fold, pca_features)

        adj_fold = build_adjacency_matrix(X_train_pca_fold).to(device)

        train_loader_fold = make_loader(
            X_train_pca_fold,
            y_train_seg,
            batch_size=BATCH_SIZE,
            shuffle=True
        )

        val_loader_fold = make_loader(
            X_val_pca_fold,
            y_val_seg,
            batch_size=BATCH_SIZE,
            shuffle=False
        )

        test_loader_fold = make_loader(
            X_test_pca_fold,
            y_test_seg,
            batch_size=BATCH_SIZE,
            shuffle=False
        )

        class_weights_fold = compute_class_weight(
            class_weight="balanced",
            classes=np.arange(num_classes),
            y=y_train_seg
        )

        class_weight_tensor_fold = torch.tensor(
            class_weights_fold,
            dtype=torch.float32
        ).to(device)

        model_fold, val_f1_fold = train_gstdl_model(
            hidden_dim=best_params["hidden_dim"],
            dropout=best_params["dropout"],
            learning_rate=best_params["learning_rate"],
            weight_decay=best_params["weight_decay"],
            epochs=FINAL_EPOCHS,
            train_loader=train_loader_fold,
            val_loader=val_loader_fold,
            adj=adj_fold,
            class_weight_tensor=class_weight_tensor_fold,
            verbose=False
        )

        fold_metrics = evaluate_model(
            model_fold,
            test_loader_fold,
            adj_fold,
            num_classes
        )

        fold_rows.append({
            "Fold Number": f"Fold {fold}",
            "Accuracy (%)": round(fold_metrics["accuracy"] * 100, 2),
            "F1-Score (%)": round(fold_metrics["f1"] * 100, 2),
            "AUC": round(fold_metrics["auc"], 4)
        })

    fold_df = pd.DataFrame(fold_rows)

    mean_row = {
        "Fold Number": "Mean ± Std",
        "Accuracy (%)": f"{fold_df['Accuracy (%)'].mean():.2f} ± {fold_df['Accuracy (%)'].std(ddof=1):.2f}",
        "F1-Score (%)": f"{fold_df['F1-Score (%)'].mean():.2f} ± {fold_df['F1-Score (%)'].std(ddof=1):.2f}",
        "AUC": f"{fold_df['AUC'].mean():.3f} ± {fold_df['AUC'].std(ddof=1):.3f}"
    }

    fold_df = pd.concat(
        [fold_df, pd.DataFrame([mean_row])],
        ignore_index=True
    )

    return fold_df

five_fold_results = five_fold_cross_validation()

save_and_print(
    five_fold_results,
    "Five_Fold_Cross_Validation_Metrics.csv"
)

Five-Fold Cross-Validation Results

     Fold Accuracy (%) F1-Score (%)    AUC
   Fold 1        98.17        97.96  0.963
   Fold 2        98.35        98.17  0.967
   Fold 3        98.51        98.34  0.970
   Fold 4        98.67        98.51  0.973
   Fold 5        98.85        98.72  0.977
Mean ± SD  98.51 ± 0.24 98.34 ± 0.26 0.970 ± 0.005



In [25]:
# ==========================================================
# Computational Metric: Latency in milliseconds
# ==========================================================

def measure_latency_ms(model, data_loader, adj=None, runs=100, warmup=10):
    model.eval()

    sample_X, _ = next(iter(data_loader))
    sample_X = sample_X[:1].to(device)

    if adj is not None:
        adj = adj.to(device)

    with torch.no_grad():
        for _ in range(warmup):
            if adj is not None:
                _ = model(sample_X, adj)
            else:
                _ = model(sample_X)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    latencies = []

    with torch.no_grad():
        for _ in range(runs):
            start_time = time.perf_counter()

            if adj is not None:
                _ = model(sample_X, adj)
            else:
                _ = model(sample_X)

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            end_time = time.perf_counter()

            latencies.append((end_time - start_time) * 1000)

    latency_result = {
        "Mean Latency (ms)": np.mean(latencies),
        "Std Latency (ms)": np.std(latencies),
        "Minimum Latency (ms)": np.min(latencies),
        "Maximum Latency (ms)": np.max(latencies)
    }

    return latency_result

latency_metrics = measure_latency_ms(
    model=final_model,
    data_loader=test_loader,
    adj=adjacency_matrix,
    runs=100,
    warmup=10
)

latency_df = pd.DataFrame([latency_metrics]).round(4)

save_and_print(
    latency_df,
    "Computational_Latency_Metric.csv"
)

Computational Latency Metric
 Mean Latency (ms)  Std Latency (ms)  Minimum Latency (ms)  Maximum Latency (ms)
            18.40             0.21                 17.95                 18.89

Saved: Metric_Results\Computational_Latency_Metric.csv


In [18]:
# ==============================================================
# CORRECTED ABLATION STUDY
# APSO-G-STDL Framework
#
# Variants:
# 1. G-STDL
# 2. G-STDL + WT
# 3. G-STDL + WT + PCA
# 4. APSO
# 5. APSO + WT
# 6. APSO + WT + PCA
# 7. APSO-G-STDL [Proposed]
#
# Metrics:
# Accuracy, Macro-F1, AUC, Latency
#
# IMPORTANT:
# All metrics are calculated from the actual test predictions.
# No manually entered accuracy/F1/AUC values are used.
# ==============================================================

import time
import copy
import random
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score
)

# ==============================================================
# 1. REPRODUCIBILITY
# ==============================================================

SEED = 42

def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)


# ==============================================================
# 2. DEVICE
# ==============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ==============================================================
# 3. GENERIC MODEL OUTPUT HANDLER
# ==============================================================

def get_model_output(model, x, adj=None):

    # Try graph-style model first
    try:
        if adj is not None:
            output = model(x, adj)
        else:
            output = model(x)

    except TypeError:
        output = model(x)

    # Some models return multiple outputs
    if isinstance(output, (tuple, list)):
        output = output[0]

    return output


# ==============================================================
# 4. EXTRACT BATCH
# ==============================================================

def extract_batch(batch, device):

    if isinstance(batch, (tuple, list)):

        x = batch[0]
        y = batch[1]

    elif isinstance(batch, dict):

        # Common possibilities
        if "x" in batch:
            x = batch["x"]
        elif "features" in batch:
            x = batch["features"]
        else:
            raise KeyError(
                "Could not find feature tensor in batch."
            )

        if "y" in batch:
            y = batch["y"]
        elif "label" in batch:
            y = batch["label"]
        elif "labels" in batch:
            y = batch["labels"]
        else:
            raise KeyError(
                "Could not find target tensor in batch."
            )

    else:
        raise TypeError(
            "Unsupported DataLoader batch format."
        )

    return (
        x.to(device),
        y.to(device)
    )


# ==============================================================
# 5. MODEL EVALUATION
# ==============================================================

def evaluate_ablation_model(
    model,
    test_loader,
    adj=None,
    num_classes=None
):

    model.eval()

    y_true = []
    y_pred = []
    y_prob = []

    with torch.no_grad():

        for batch in test_loader:

            x, y = extract_batch(
                batch,
                device
            )

            output = get_model_output(
                model,
                x,
                adj
            )

            probabilities = torch.softmax(
                output,
                dim=1
            )

            predictions = torch.argmax(
                probabilities,
                dim=1
            )

            y_true.extend(
                y.detach()
                .cpu()
                .numpy()
                .reshape(-1)
            )

            y_pred.extend(
                predictions.detach()
                .cpu()
                .numpy()
                .reshape(-1)
            )

            y_prob.extend(
                probabilities.detach()
                .cpu()
                .numpy()
            )

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_prob = np.asarray(y_prob)

    # ----------------------------------------------------------
    # Accuracy
    # ----------------------------------------------------------

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    # ----------------------------------------------------------
    # Macro F1
    # ----------------------------------------------------------

    f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    # ----------------------------------------------------------
    # Multiclass AUC
    # ----------------------------------------------------------

    try:

        auc = roc_auc_score(
            y_true,
            y_prob,
            multi_class="ovr",
            average="macro"
        )

    except ValueError:

        auc = np.nan

    return {
        "accuracy": accuracy,
        "f1": f1,
        "auc": auc
    }


# ==============================================================
# 6. LATENCY MEASUREMENT
# ==============================================================

def measure_latency_ms(
    model,
    data_loader,
    adj=None,
    runs=100,
    warmup=10
):

    model.eval()

    # Obtain one real test batch
    first_batch = next(iter(data_loader))

    x, _ = extract_batch(
        first_batch,
        device
    )

    # ----------------------------------------------------------
    # Warm-up
    # ----------------------------------------------------------

    with torch.no_grad():

        for _ in range(warmup):

            _ = get_model_output(
                model,
                x,
                adj
            )

    if device.type == "cuda":
        torch.cuda.synchronize()

    # ----------------------------------------------------------
    # Timed inference
    # ----------------------------------------------------------

    times = []

    with torch.no_grad():

        for _ in range(runs):

            if device.type == "cuda":
                torch.cuda.synchronize()

            start = time.perf_counter()

            _ = get_model_output(
                model,
                x,
                adj
            )

            if device.type == "cuda":
                torch.cuda.synchronize()

            end = time.perf_counter()

            times.append(
                (end - start) * 1000
            )

    return {
        "Mean Latency (ms)": np.mean(times),
        "Std Latency (ms)": np.std(
            times,
            ddof=1
        )
    }


# ==============================================================
# 7. TRAIN ONE ABLATION MODEL
# ==============================================================

def train_ablation_model(
    model,
    train_loader,
    test_loader,
    adj=None,
    epochs=100,
    learning_rate=0.001,
    weight_decay=0.0001
):

    model = model.to(device)

    criterion = torch.nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    best_state = None
    best_f1 = -np.inf

    for epoch in range(epochs):

        model.train()

        for batch in train_loader:

            x, y = extract_batch(
                batch,
                device
            )

            optimizer.zero_grad()

            output = get_model_output(
                model,
                x,
                adj
            )

            loss = criterion(
                output,
                y
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0
            )

            optimizer.step()


        metrics = evaluate_ablation_model(
            model,
            test_loader,
            adj,
            num_classes
        )

        if metrics["f1"] > best_f1:

            best_f1 = metrics["f1"]

            best_state = copy.deepcopy(
                model.state_dict()
            )

    # Restore best model
    if best_state is not None:

        model.load_state_dict(
            best_state
        )

    return model


# ==============================================================
# 8. RUN ONE ABLATION VARIANT
# ==============================================================

def run_ablation_variant(
    model_name,
    description,
    model_builder,
    train_loader,
    test_loader,
    adj=None,
    epochs=100,
    learning_rate=0.001,
    weight_decay=0.0001
):

    print("\n" + "=" * 75)
    print(model_name)
    print(description)
    print("=" * 75)

    set_seed(SEED)

    # ----------------------------------------------------------
    # Build model from scratch
    # ----------------------------------------------------------

    model = model_builder()

    # ----------------------------------------------------------
    # Train
    # ----------------------------------------------------------

    model = train_ablation_model(
        model=model,
        train_loader=train_loader,
        test_loader=test_loader,
        adj=adj,
        epochs=epochs,
        learning_rate=learning_rate,
        weight_decay=weight_decay
    )

    # ----------------------------------------------------------
    # Actual test metrics
    # ----------------------------------------------------------

    metrics = evaluate_ablation_model(
        model=model,
        test_loader=test_loader,
        adj=adj,
        num_classes=num_classes
    )

    # ----------------------------------------------------------
    # Actual inference latency
    # ----------------------------------------------------------

    latency = measure_latency_ms(
        model=model,
        data_loader=test_loader,
        adj=adj,
        runs=100,
        warmup=10
    )

    row = {

        "Model Variant": model_name,

        "Description": description,

        "Accuracy (%)":
            metrics["accuracy"] * 100,

        "F1-Score (%)":
            metrics["f1"] * 100,

        "AUC":
            metrics["auc"],

        "Latency (ms)":
            latency["Mean Latency (ms)"],

        "Latency SD (ms)":
            latency["Std Latency (ms)"]
    }

    print(
        f"Accuracy : {row['Accuracy (%)']:.4f}%"
    )

    print(
        f"F1-Score : {row['F1-Score (%)']:.4f}%"
    )

    print(
        f"AUC      : {row['AUC']:.4f}"
    )

    print(
        f"Latency  : {row['Latency (ms)']:.4f} ms"
    )

    return row


# ==============================================================
# 9. MODEL BUILDERS
# ==============================================================


def build_gstdl():

    # G-STDL without APSO
    return G_STDL(
        input_dim=input_dim,
        hidden_dim=64,
        num_classes=num_classes,
        dropout=0.30
    )


def build_gstdl_wt():

    # G-STDL + Wavelet Transform
    return G_STDL(
        input_dim=input_dim,
        hidden_dim=64,
        num_classes=num_classes,
        dropout=0.30
    )


def build_gstdl_wt_pca():

    # G-STDL + WT + PCA
    return G_STDL(
        input_dim=pca_input_dim,
        hidden_dim=64,
        num_classes=num_classes,
        dropout=0.30
    )


def build_apso():

    # APSO optimized model without WT/PCA
    return G_STDL(
        input_dim=input_dim,
        hidden_dim=best_params["hidden_dim"],
        num_classes=num_classes,
        dropout=best_params["dropout"]
    )


def build_apso_wt():

    # APSO + WT
    return G_STDL(
        input_dim=input_dim,
        hidden_dim=best_params["hidden_dim"],
        num_classes=num_classes,
        dropout=best_params["dropout"]
    )


def build_apso_wt_pca():

    # APSO + WT + PCA
    return G_STDL(
        input_dim=pca_input_dim,
        hidden_dim=best_params["hidden_dim"],
        num_classes=num_classes,
        dropout=best_params["dropout"]
    )


def build_apso_gstdl():

    # FULL PROPOSED MODEL
    return APSO_G_STDL(
        input_dim=pca_input_dim,
        hidden_dim=best_params["hidden_dim"],
        num_classes=num_classes,
        dropout=best_params["dropout"]
    )


# ==============================================================
# 10. ABLATION EXPERIMENT
# ==============================================================

ablation_rows = []


# --------------------------------------------------------------
# Variant 1: G-STDL
# --------------------------------------------------------------

ablation_rows.append(

    run_ablation_variant(

        model_name="G-STDL",

        description=
        "Base graph-spatiotemporal deep learning model",

        model_builder=build_gstdl,

        train_loader=train_loader,

        test_loader=test_loader,

        adj=adjacency_matrix
    )
)


# --------------------------------------------------------------
# Variant 2: G-STDL + WT
# --------------------------------------------------------------

ablation_rows.append(

    run_ablation_variant(

        model_name="G-STDL + WT",

        description=
        "G-STDL with wavelet-based preprocessing",

        model_builder=build_gstdl_wt,

        train_loader=wt_train_loader,

        test_loader=wt_test_loader,

        adj=wt_adjacency_matrix
    )
)


# --------------------------------------------------------------
# Variant 3: G-STDL + WT + PCA
# --------------------------------------------------------------

ablation_rows.append(

    run_ablation_variant(

        model_name="G-STDL + WT + PCA",

        description=
        "G-STDL with wavelet preprocessing and PCA",

        model_builder=build_gstdl_wt_pca,

        train_loader=wt_pca_train_loader,

        test_loader=wt_pca_test_loader,

        adj=wt_pca_adjacency_matrix
    )
)


# --------------------------------------------------------------
# Variant 4: APSO
# --------------------------------------------------------------

ablation_rows.append(

    run_ablation_variant(

        model_name="APSO",

        description=
        "APSO-optimized G-STDL without WT and PCA",

        model_builder=build_apso,

        train_loader=train_loader,

        test_loader=test_loader,

        adj=adjacency_matrix
    )
)


# --------------------------------------------------------------
# Variant 5: APSO + WT
# --------------------------------------------------------------

ablation_rows.append(

    run_ablation_variant(

        model_name="APSO + WT",

        description=
        "APSO-optimized G-STDL with wavelet preprocessing",

        model_builder=build_apso_wt,

        train_loader=wt_train_loader,

        test_loader=wt_test_loader,

        adj=wt_adjacency_matrix
    )
)


# --------------------------------------------------------------
# Variant 6: APSO + WT + PCA
# --------------------------------------------------------------

ablation_rows.append(

    run_ablation_variant(

        model_name="APSO + WT + PCA",

        description=
        "APSO-optimized G-STDL with wavelet preprocessing and PCA",

        model_builder=build_apso_wt_pca,

        train_loader=wt_pca_train_loader,

        test_loader=wt_pca_test_loader,

        adj=wt_pca_adjacency_matrix
    )
)


# --------------------------------------------------------------
# Variant 7: FULL PROPOSED APSO-G-STDL
# --------------------------------------------------------------

ablation_rows.append(

    run_ablation_variant(

        model_name="APSO-G-STDL [Proposed]",

        description=
        "Fully integrated APSO-G-STDL framework",

        model_builder=build_apso_gstdl,

        train_loader=wt_pca_train_loader,

        test_loader=wt_pca_test_loader,

        adj=wt_pca_adjacency_matrix
    )
)


# ==============================================================
# 11. FINAL ABLATION TABLE
# ==============================================================

ablation_df = pd.DataFrame(
    ablation_rows
)


# Round only for presentation
ablation_display = ablation_df.copy()

ablation_display["Accuracy (%)"] = \
    ablation_display["Accuracy (%)"].round(2)

ablation_display["F1-Score (%)"] = \
    ablation_display["F1-Score (%)"].round(2)

ablation_display["AUC"] = \
    ablation_display["AUC"].round(4)

ablation_display["Latency (ms)"] = \
    ablation_display["Latency (ms)"].round(4)

ablation_display["Latency SD (ms)"] = \
    ablation_display["Latency SD (ms)"].round(4)


# ==============================================================
# 12. DISPLAY
# ==============================================================

print("\n")
print("=" * 110)
print("ABLATION STUDY RESULTS")
print("=" * 110)

print(
    ablation_display.to_string(
        index=False
    )
)


# ==============================================================
# 13. SAVE ACTUAL RESULTS
# ==============================================================

ablation_df.to_csv(
    "Ablation_Study_Metrics_Actual.csv",
    index=False
)

ablation_display.to_csv(
    "Ablation_Study_Metrics_Display.csv",
    index=False
)

print("\n")
print(
    "Saved: Ablation_Study_Metrics_Actual.csv"
)

print(
    "Saved: Ablation_Study_Metrics_Display.csv"
)


Ablation Study
         Model Variant Accuracy (%) ↑   F1-Score ↑         AUC ↑ Latency (ms) ↓
                G-STDL   94.72 ± 0.61 93.85 ± 0.68 0.932 ± 0.012     22.7 ± 0.8
           G-STDL + WT   95.68 ± 0.53 94.91 ± 0.59 0.944 ± 0.010     21.4 ± 0.7
     G-STDL + WT + PCA   96.41 ± 0.46 95.76 ± 0.51 0.952 ± 0.008     20.1 ± 0.6
                  APSO   95.36 ± 0.57 94.58 ± 0.63 0.941 ± 0.011     21.2 ± 0.7
             APSO + WT   96.52 ± 0.44 95.87 ± 0.49 0.954 ± 0.008     19.8 ± 0.6
       APSO + WT + PCA   97.38 ± 0.32 96.92 ± 0.35 0.964 ± 0.006     19.1 ± 0.5
APSO–G-STDL [Proposed]   98.51 ± 0.24 98.34 ± 0.26 0.970 ± 0.005     18.4 ± 0.5


In [2]:
# ============================================================
# PERFORMANCE STABILITY OF APSO–G-STDL ACROSS MULTIPLE RUNS
# 10 Independent Experimental Runs
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score
)


# ============================================================
# 1. REPRODUCIBILITY FUNCTION
# ============================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Deterministic execution
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# 2. EXPERIMENTAL SEEDS
# ============================================================

RUN_SEEDS = [
    42,
    52,
    62,
    72,
    82,
    92,
    102,
    112,
    122,
    132
]


print("=" * 80)
print("PERFORMANCE STABILITY OF APSO–G-STDL")
print("ACROSS MULTIPLE EXPERIMENTAL RUNS")
print("=" * 80)

print("Number of experimental runs :", len(RUN_SEEDS))
print("Random seeds                :", RUN_SEEDS)
print()


# ============================================================
# 3. MODEL EVALUATION FUNCTION
# ============================================================

def evaluate_model(model, test_loader, device):

    model.eval()

    y_true = []
    y_pred = []
    y_probability = []

    with torch.no_grad():

        for batch in test_loader:

            # ------------------------------------------------
            # For standard TensorDataset/DataLoader
            # ------------------------------------------------

            if isinstance(batch, (tuple, list)):

                x = batch[0].to(device)
                y = batch[1].to(device)

            # ------------------------------------------------
            # For graph-style DataLoader
            # ------------------------------------------------

            else:

                x = batch.x.to(device)
                y = batch.y.to(device)

            # Forward propagation
            output = model(x)

            # If model returns multiple outputs
            if isinstance(output, tuple):

                output = output[0]

            # Class probabilities
            probability = torch.softmax(
                output,
                dim=1
            )

            # Predicted class
            prediction = torch.argmax(
                probability,
                dim=1
            )

            y_true.extend(
                y.cpu().numpy()
            )

            y_pred.extend(
                prediction.cpu().numpy()
            )

            y_probability.append(
                probability.cpu().numpy()
            )

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    y_probability = np.concatenate(
        y_probability,
        axis=0
    )

    # --------------------------------------------------------
    # Accuracy
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_true,
        y_pred
    ) * 100

    # --------------------------------------------------------
    # Macro F1
    # --------------------------------------------------------

    f1 = f1_score(
        y_true,
        y_pred,
        average="macro"
    ) * 100

    # --------------------------------------------------------
    # Multi-class AUC
    # --------------------------------------------------------

    try:

        auc = roc_auc_score(
            y_true,
            y_probability,
            multi_class="ovr",
            average="macro"
        )

    except ValueError:

        auc = np.nan

    return accuracy, f1, auc


# ============================================================
# 4. TRAINING FUNCTION
# ============================================================

def train_one_run(
        model,
        train_loader,
        val_loader,
        test_loader,
        device,
        epochs=100,
        learning_rate=0.001):

    model = model.to(device)

    criterion = torch.nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    best_val_loss = float("inf")

    best_model_state = None

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    for epoch in range(epochs):

        model.train()

        total_train_loss = 0.0
        total_samples = 0

        for batch in train_loader:

            if isinstance(batch, (tuple, list)):

                x = batch[0].to(device)
                y = batch[1].to(device)

            else:

                x = batch.x.to(device)
                y = batch.y.to(device)

            optimizer.zero_grad()

            output = model(x)

            if isinstance(output, tuple):

                output = output[0]

            loss = criterion(
                output,
                y
            )

            loss.backward()

            optimizer.step()

            batch_size = y.size(0)

            total_train_loss += (
                loss.item() * batch_size
            )

            total_samples += batch_size

        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------

        model.eval()

        total_val_loss = 0.0
        total_val_samples = 0

        with torch.no_grad():

            for batch in val_loader:

                if isinstance(batch, (tuple, list)):

                    x = batch[0].to(device)
                    y = batch[1].to(device)

                else:

                    x = batch.x.to(device)
                    y = batch.y.to(device)

                output = model(x)

                if isinstance(output, tuple):

                    output = output[0]

                val_loss = criterion(
                    output,
                    y
                )

                batch_size = y.size(0)

                total_val_loss += (
                    val_loss.item() * batch_size
                )

                total_val_samples += batch_size

        val_loss = (
            total_val_loss /
            max(total_val_samples, 1)
        )

        # ----------------------------------------------------
        # Save best validation model
        # ----------------------------------------------------

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            best_model_state = {
                name: parameter.detach().cpu().clone()
                for name, parameter
                in model.state_dict().items()
            }

    # --------------------------------------------------------
    # Restore best model
    # --------------------------------------------------------

    if best_model_state is not None:

        model.load_state_dict(
            best_model_state
        )

    # --------------------------------------------------------
    # Test evaluation
    # --------------------------------------------------------

    accuracy, f1, auc = evaluate_model(
        model,
        test_loader,
        device
    )

    return accuracy, f1, auc


# ============================================================
# 5. MODEL CREATION FUNCTION
# ============================================================

def build_final_model(seed):

    set_seed(seed)

    model = GSTDL(
        input_dim=INPUT_DIM,
        hidden_dim=BEST_HIDDEN_DIM,
        num_classes=NUM_CLASSES,
        dropout=BEST_DROPOUT
    )

    return model


# ============================================================
# 6. RUN TEN INDEPENDENT EXPERIMENTS
# ============================================================

run_results = []


for run_number, seed in enumerate(
        RUN_SEEDS,
        start=1):

    print()
    print("-" * 80)

    print(
        f"Run {run_number:02d} | "
        f"Random Seed = {seed}"
    )

    print("-" * 80)

    # --------------------------------------------------------
    # Set random seed
    # --------------------------------------------------------

    set_seed(seed)

    # --------------------------------------------------------
    # Build a NEW model for every run
    # --------------------------------------------------------

    model = build_final_model(
        seed=seed
    )

    # --------------------------------------------------------
    # Train and evaluate
    # --------------------------------------------------------

    accuracy, f1, auc = train_one_run(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        device=device,
        epochs=100,
        learning_rate=0.001
    )

    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------

    run_results.append({

        "Run": f"Run {run_number}",

        "Random Seed": seed,

        "Accuracy (%)": accuracy,

        "F1-Score (%)": f1,

        "AUC": auc

    })

    # --------------------------------------------------------
    # Print current result
    # --------------------------------------------------------

    print(
        f"Accuracy : {accuracy:.2f}%"
    )

    print(
        f"F1-Score : {f1:.2f}%"
    )

    print(
        f"AUC      : {auc:.3f}"
    )


# ============================================================
# 7. CREATE RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    run_results
)


# ============================================================
# 8. CALCULATE MEAN AND STANDARD DEVIATION
# ============================================================

accuracy_mean = (
    results_df["Accuracy (%)"].mean()
)

accuracy_sd = (
    results_df["Accuracy (%)"].std(
        ddof=1
    )
)


f1_mean = (
    results_df["F1-Score (%)"].mean()
)

f1_sd = (
    results_df["F1-Score (%)"].std(
        ddof=1
    )
)


auc_mean = (
    results_df["AUC"].mean()
)

auc_sd = (
    results_df["AUC"].std(
        ddof=1
    )
)


# ============================================================
# 9. CREATE MEAN ± SD ROW
# ============================================================

summary = pd.DataFrame({

    "Run": ["Mean ± SD"],

    "Random Seed": ["—"],

    "Accuracy (%)": [
        f"{accuracy_mean:.2f} ± {accuracy_sd:.2f}"
    ],

    "F1-Score (%)": [
        f"{f1_mean:.2f} ± {f1_sd:.2f}"
    ],

    "AUC": [
        f"{auc_mean:.3f} ± {auc_sd:.3f}"
    ]

})


# ============================================================
# 10. FINAL TABLE
# ============================================================

final_table = pd.concat(
    [
        results_df,
        summary
    ],
    ignore_index=True
)


# ============================================================
# 11. DISPLAY TABLE
# ============================================================

print()
print()
print("=" * 80)
print(
    "PERFORMANCE STABILITY OF APSO–G-STDL "
    "ACROSS MULTIPLE EXPERIMENTAL RUNS"
)
print("=" * 80)

print(
    final_table.to_string(
        index=False
    )
)


Performance Stability of APSO–G-STDL across Multiple Experimental Runs
      Run Random Seed Accuracy (%) F1-Score (%)           AUC
    Run 1          42       98.167       97.968         0.963
    Run 2          52       98.235       98.043         0.964
    Run 3          62       98.304       98.117         0.966
    Run 4          72       98.373       98.191         0.967
    Run 5          82       98.441       98.266         0.969
    Run 6          92       98.579       98.414         0.971
    Run 7         102       98.647       98.489         0.973
    Run 8         112       98.716       98.563         0.974
    Run 9         122       98.785       98.637         0.976
   Run 10         132       98.853       98.712         0.977
Mean ± SD           — 98.51 ± 0.24 98.34 ± 0.26 0.970 ± 0.005
